# LSI-lite (Colab) — Figure / Mark-Making / Landscape (+ Color and Tonal Analysis)

**Small, profiled composition gate for quick QA of images.
LSI-lite measures how an image behaves under compositional structure using three primitives: Δx (off-center gravity), rᵥ (void ratio), ρᵣ (rupture/mark energy) and tells you if it sits within intended bands for its class. It's built to study stability, not to crown winners.**

- Balance (Δx): How far the visual center is from the geometric center
- Density (rᵥ): The ratio of empty space to filled space
- Detail (ρᵣ): The amount of edge energy and texture density in key areas

It combines these measurements into a 0-100 score for how an image lines up or "passes" basic structural compositional criteria. It helps distinguish delta in AI and human default.

**Quick Start**

**Run Setup & Config**
- Installs requirements and sets defaults. (No need to tweak pins unless you want custom paths.)
- (Optional) Enable Color/Tonal analysis (enabled by default)
- By default, color checks are ON with COLOR_SPACE="LAB". Turn off by setting INCLUDE_COLOR=False.
- Upload your images
- Choose Profile mode — use Auto for ease or manually select
- Run Scoring — The notebook computes Δx, rᵥ, ρᵣ (and color/tonal if enabled). Tables show per-image scores and pass/fail badges.
- Plots visualize centroid drift, void ratio, and stroke energy. Audit badges (color/tonal) only appear if thresholds trip.


Profiles
Figure_Default, MarkMaking_Expressive, Landscape — each has custom band guards + weights.

**How Color Analysis Works**

Color analysis operates in two modes:

**1. Mask Selection (Can Affect Score)**
- LAB k-means (k=3) creates a color-based foreground mask
- Largest cluster → background; foreground = union of others + morphology
- If the color mask has better subject isolation (based on area fraction 0.03–0.65), it replaces the grayscale mask
- When color mask is selected, its Δx and rᵥ values replace the gray-based values in the final LSI score
- This means color can influence your pass/fail result through better subject detection
- Fallback: if color fails or produces poor masks, grayscale is used automatically

**2. Telemetry & Audit (Advisory Only)**

Color and tonal metrics provide diagnostic information but don't directly gate images:

Color telemetry:
- rv_color_mask, dx_color_L — measurements using color mask
- delta_rv, delta_dx — differences between color and gray measurements
- mask_mode — which mask was used: gray | color | external
- color_status — ok | fallback_gray | fail_convert
- color_audit_badge — fires if large disagreement (|Δrᵥ| ≥ 0.85 or |Δx| ≥ 0.20)

Tonal telemetry (L-channel analysis, ROI-aligned with Δx):

- S_L — tonal span (P95 − P5) on L-channel, normalized [0–1]
- eta_L — Otsu separability [0–1], measures tonal separation
- beta_L — bright mass fraction (L ≥ 0.80)
 -tonal_audit_badge — fires if flat/washed out (beta_L < 0.12)

Note: ρᵣ (rupture) is always computed from grayscale using the halo mask, regardless of color settings.

**Optional External Masks**

You can provide external uint8 masks (0/255) via mask_mode='external' to override both gray and color detection.

**Landscape-Specific Features**

When using Landscape profile with dx_roi="auto":
- Auto-detects water reflections using edge similarity and horizontal gradients
- Crops Δx measurement to top 60% to avoid reflection interference
- Tonal metrics use the same ROI for consistency



This is not recognition or a "style police" or a judgment on "aesthetics." It's a tiny, defensible ruler over three compositional 101 primitives with color and tonal-aware analysis to clarify why a frame passes or fails. Think of it as a quick ruler for balance, void, and stroke coherence. LSI-lite as a complementary metric in the generative AI evaluation ecosystem.

In [ ]:
# @title Setup & Config (resilient, pinned when requested)
PIN_DEPS = False  # @param {type:"boolean"}

PINS = {
    "opencv-python-headless": "4.10.0.84",
    "numpy": "1.26.4",
    "pandas": "2.0.3",
    "matplotlib": "3.7.5",
    "Pillow": "10.4.0",
}

def maybe_pin_deps(pin=PIN_DEPS):
    """Install pinned versions only if requested; otherwise use Colab defaults."""
    if not pin:
        print("Pinned deps disabled. Using environment defaults.")
        return
    import sys, subprocess
    pkgs = [f"{k}=={v}" for k, v in PINS.items()]
    cmd = [sys.executable, "-m", "pip", "install",
           "--prefer-binary", "--only-if-needed", "--upgrade-strategy", "only-if-needed"] + pkgs
    print("Installing pinned:", " ".join(pkgs))
    try:
        subprocess.check_call(cmd)
        print("Pin install finished.")
    except Exception as e:
        print("Pin install failed; continuing with existing env:", e)

maybe_pin_deps()

# now the normal imports
import os, glob, math, io, json, shutil, base64
import random
import numpy as np, cv2, pandas as pd
np.random.seed(0)
random.seed(0)
cv2.setRNGSeed(0)
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

os.makedirs("/content/images", exist_ok=True)

CONFIG = {
    "preprocessing": {"longest_side": 1536, "morph_kernel": 5},
    "accept": {"gate_100": 55.0},
    "sigma_scale": 0.35,
}

PROFILES = {
    "Figure_Default": {
        "weights": {"dx": 0.45, "rv": 0.35, "rho": 0.20},
        "bands": {
            "dx":  {"guard": [0.05, 0.85]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.10, 0.80]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.08},
        "dx_roi": None,
    },
    "MarkMaking_Expressive": {
        "weights": {"dx": 0.40, "rv": 0.30, "rho": 0.30},
        "bands": {
            "dx":  {"guard": [0.02, 0.90]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.06, 0.70]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.12},  # ← keep only this one
        "dx_roi": None,
    },
    "Landscape": {
        "weights": {"dx": 0.35, "rv": 0.40, "rho": 0.25},
        "bands": {
            "dx":  {"guard": [0.05, 0.95]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.08, 0.80]},
        },
        "rho_mask": {"type": "full"},
        "dx_roi": "auto",   # ← turn on reflection-aware Δx crop (or set to None to disable)
    },
}
print("Ready. Profiles:", list(PROFILES.keys()))
# --- Color telemetry toggles (does NOT affect gating) ---
INCLUDE_COLOR = True      # flip to False to disable all color telemetry
COLOR_SPACE   = "LAB"     # LAB (OpenCV) or OKLab if you add it later

COLOR_TELEMETRY_FIELDS = [
    # color telemetry
    "rv_color_mask","dx_color_L","delta_rv","delta_dx",
    "mask_mode","color_space","color_status","color_audit_badge",
    # tonal telemetry
    "S_L","eta_L","beta_L","tonal_status","tonal_audit_badge",
]

In [ ]:
# @title Helpers (robust loader, safe largest, morphology, masks, auto-profile)

def load_image_robust(path_or_bytes):
    """Open as RGB with EXIF orientation; return grayscale uint8 and original RGB HxWx3."""
    if isinstance(path_or_bytes, (bytes, bytearray)):
        img = Image.open(io.BytesIO(path_or_bytes))
    else:
        img = Image.open(path_or_bytes)
    img = ImageOps.exif_transpose(img).convert("RGB")
    rgb = np.asarray(img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return gray, rgb

def resize_longest(gray, longest):
    h, w = gray.shape[:2]
    s = longest / max(h, w)
    if s < 1.0:
        gray = cv2.resize(gray, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
    return gray

def safe_largest(bin_u8: np.ndarray):
    cnts, _ = cv2.findContours(bin_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    a = max(cnts, key=cv2.contourArea)
    mask = np.zeros_like(bin_u8, dtype=np.uint8)
    cv2.drawContours(mask, [a], -1, 255, -1)
    return mask

def _morph(mask, k):
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    return mask

def foreground_mask(gray: np.ndarray, cfg: dict) -> np.ndarray:
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, bin_ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Prefer “dark subject on light paper”, then try normal, then degenerate fallback
    mask = safe_largest((255 - bin_).astype(np.uint8))
    if mask is None:
        mask = safe_largest(bin_.astype(np.uint8))
    if mask is None:
        mask = (bin_ > 0).astype(np.uint8) * 255

    # Light clean-up
    k = cfg["preprocessing"]["morph_kernel"]
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, 1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, 1)
    return mask

def subject_halo_mask(fg_mask, halo_frac=0.10):
    h, w = fg_mask.shape[:2]
    r = max(1, int(halo_frac * min(h, w)))
    kernel = np.ones((r, r), np.uint8)
    halo = cv2.dilate(fg_mask, kernel, iterations=1)
    return halo

def auto_profile(dx, rv):
    # very small dx + high rv => Landscape; mid dx / mid rv => Figure; else MarkMaking
    if rv >= 0.35 and dx <= 0.25:
        return "Landscape"
    if rv <= 0.55 and dx >= 0.15:
        return "Figure_Default"
    return "MarkMaking_Expressive"

def band_center_and_span(profile: str, band: str):
    """Return (center, half-span) from the guard band for a given profile/band."""
    g = PROFILES[profile]["bands"][band]["guard"]  # [lo, hi]
    c = 0.5 * (g[0] + g[1])
    s = max(1e-6, 0.5 * (g[1] - g[0]))
    return c, s

def ensure_gray_array(x):
    """Accept path | ndarray | tuple and return a 2-D uint8 grayscale array."""
    # If it's a path/bytes, load it
    if isinstance(x, (str, bytes, bytearray)):
        g = load_image_robust(x)
    else:
        g = x

    # Some code paths hand us (gray, extra). Take the first item.
    if isinstance(g, tuple):
        g = g[0]

    g = np.asarray(g)
    # If RGB/BGR, convert to gray
    if g.ndim == 3 and g.shape[2] in (3, 4):
        # assume RGB because load_image_robust returns RGB
        g = cv2.cvtColor(g, cv2.COLOR_RGB2GRAY)
    # Ensure uint8
    if g.dtype != np.uint8:
        g = np.clip(g, 0, 255).astype(np.uint8)
    return g

def auto_dx_roi(gray_or_tuple,
                *,
                top_frac: float = 0.60,
                reflect_thr: float = 0.35,     # was 0.50; looser to catch real lakes
                waterish_ratio: float = 1.15,  # was 1.30; horizontal > vertical
                require_symmetry: bool = False,
                sym_eps: float = 0.25):
    """
    Auto-crop ROI for Δx when a water reflection is present (Landscape).
    Accepts path/ndarray/(gray,rgb) tuple. Returns dict {"type":"top_frac","top":<f>}
    or None when no crop is recommended.

    Heuristic:
      1) Reflection similarity between top half and flipped bottom half (edges).
      2) "Water-ish" bottom: horizontal > vertical gradients.
      3) (optional) Similar foreground fill in top/bottom halves.
    """
    # 1) Get a 2-D uint8 grayscale image, regardless of input type
    g = ensure_gray_array(gray_or_tuple)   # your helper: path/tuple-safe → 2-D uint8
    if g is None or g.ndim != 2:
        return None
    H, W = g.shape
    if H < 4 or W < 4:
        return None

    h2  = H // 2
    top = g[:h2, :]
    bot = g[h2:, :]

    # 2) Edge maps for similarity probe
    e_top = cv2.Canny(top, 50, 100).astype(np.float32)
    e_bot = cv2.Canny(bot, 50, 100).astype(np.float32)
    e_bot_flip = np.flipud(e_bot)

    # Cosine similarity between the two halves’ edge maps
    num = (e_top * e_bot_flip).sum()
    den = float(np.linalg.norm(e_top) * np.linalg.norm(e_bot_flip) + 1e-6)
    reflect_sim = num / den

    # 3) "Water-ish" check: horizontal structure should dominate in the bottom half
    gx = np.abs(cv2.Sobel(bot, cv2.CV_32F, 1, 0, ksize=3)).mean()
    gy = np.abs(cv2.Sobel(bot, cv2.CV_32F, 0, 1, ksize=3)).mean()
    waterish = gx > (waterish_ratio * gy)

    # 4) Optional symmetric foreground fill (uses your existing CONFIG + mask helper)
    ok_symmetry = True
    if require_symmetry:
        mask = foreground_mask(g, CONFIG)       # relies on global CONFIG (as in your notebook)
        fill_top = (mask[:h2, :] > 0).mean()
        fill_bot = (mask[h2:, :] > 0).mean()
        ok_symmetry = abs(fill_top - fill_bot) < sym_eps

    # 5) Decision
    if (reflect_sim > reflect_thr) and waterish and ok_symmetry:
        return {"type": "top_frac", "top": float(top_frac)}
    return None

In [ ]:
# Color helpers (telemetry only)

import cv2, numpy as np

def _largest_component_uint8(mask_u8: np.ndarray) -> np.ndarray:
    """Return only the largest connected component from a binary uint8 mask."""
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num <= 1:
        return mask_u8
    # skip label 0 (background); take argmax by area
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    keep = (labels == largest_idx).astype(np.uint8) * 255
    return keep

def color_mask_via_lab_kmeans(rgb_u8: np.ndarray, k: int = 3, downsample: int = 2, morph_kernel: int = 5) -> np.ndarray:
    """
    Build a foreground mask from color using LAB k-means (k=3).
    Heuristic: largest cluster = background; foreground = union of the rest.
    """
    if rgb_u8 is None:
        raise ValueError("color_mask_via_lab_kmeans: rgb_u8 is None")
    h, w = rgb_u8.shape[:2]
    # Convert to LAB
    lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
    # Downsample for faster k-means
    if downsample > 1:
        lab_small = cv2.resize(lab, (w // downsample, h // downsample), interpolation=cv2.INTER_AREA)
    else:
        lab_small = lab
    Z = lab_small.reshape((-1, 3)).astype(np.float32)

    # K-means
    cv2.setRNGSeed(0)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    attempts = 3
    flags = cv2.KMEANS_PP_CENTERS
    compactness, labels, centers = cv2.kmeans(Z, k, None, criteria, attempts, flags)

    labels = labels.reshape(lab_small.shape[:2])
    # Majority cluster → background
    bg_label = np.bincount(labels.flatten()).argmax()
    fg_small = (labels != bg_label).astype(np.uint8) * 255

    # Upsample to original size if needed
    if downsample > 1:
        fg = cv2.resize(fg_small, (w, h), interpolation=cv2.INTER_NEAREST)
    else:
        fg = fg_small

    # Morphology cleanup
    K = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel))
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, K)
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, K)

    # Keep largest component only
    fg = _largest_component_uint8(fg)
    return fg

def compute_rv_from_mask(mask_u8: np.ndarray) -> float:
    """rv = 1 - fill, where fill is fraction of foreground (mask>0)."""
    total = mask_u8.size
    fill = float(np.count_nonzero(mask_u8)) / float(total)
    return max(0.0, min(1.0, 1.0 - fill))

def compute_dx_color(gray_L_u8: np.ndarray, mask_u8: np.ndarray) -> float:
    """
    Δx on L-channel: edge-first centroid restricted by color-foreground mask; fallback to mask centroid.
    Returns normalized horizontal distance [0..1] or np.nan if nothing usable.
    """
    H, W = gray_L_u8.shape[:2]
    # Edge-first within mask
    edges = cv2.Canny(gray_L_u8, 50, 100)
    edges = cv2.bitwise_and(edges, edges, mask=mask_u8)
    if np.count_nonzero(edges) > 0:
        m = cv2.moments(edges)
        if m["m00"] != 0:
            cx = m["m10"] / m["m00"]
            dx = abs(cx - (W / 2.0)) / (W / 2.0)
            return float(max(0.0, min(1.0, dx)))
    # Fallback: mask centroid
    if np.count_nonzero(mask_u8) > 0:
        m = cv2.moments(mask_u8)
        if m["m00"] != 0:
            cx = m["m10"] / m["m00"]
            dx = abs(cx - (W / 2.0)) / (W / 2.0)
            return float(max(0.0, min(1.0, dx)))
    return float("nan")

def color_telemetry(gray_u8: np.ndarray, rgb_u8: np.ndarray, morph_kernel: int = 5):
    """
    Compute (rv_color, dx_color, mask_mode, color_status).
    Never raises; on any failure, returns (None, None, 'none', 'error').
    """
    try:
        if rgb_u8 is None:
            return None, None, "none", "error"
        # Use L from LAB for Δx_color
        lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
        L = lab[:, :, 0]
        # Build color-foreground mask and compute metrics
        fg_color = color_mask_via_lab_kmeans(rgb_u8, k=3, morph_kernel=morph_kernel)
        rv_c = compute_rv_from_mask(fg_color)
        dx_c = compute_dx_color(L, fg_color)
        return fg_color, dx_c, rv_c, "color", "ok"
    except Exception:
        return None, None, None, "none", "error"

def choose_subject_mask(gray_mask, color_mask, area_lo=0.03, area_hi=0.65, prefer_color=True):
    """
    Choose between gray and color masks based on quality metrics.

    Strategy:
    1. Both masks must be in valid area range [area_lo, area_hi]
    2. If only one is valid, use that one
    3. If both valid, compare quality:
       - Border touching (fewer is better)
       - Compactness (area/perimeter ratio)
       - Fill ratio distance from extremes
    4. If prefer_color=True, color wins ties

    Returns: (mask, source_name)
    """
    h, w = gray_mask.shape
    total = h * w

    def area_fraction(m):
        return (m > 0).sum() / total

    def mask_quality(m):
        """Return quality score [0-1], higher is better."""
        if m is None:
            return 0.0

        # Area fraction
        fill = area_fraction(m)

        # Border touching penalty
        touches = {
            "top": (m[0, :] > 0).any(),
            "bottom": (m[-1, :] > 0).any(),
            "left": (m[:, 0] > 0).any(),
            "right": (m[:, -1] > 0).any(),
        }
        border_penalty = sum(touches.values()) / 4.0  # 0 = no borders, 1 = all borders

        # Compactness (ratio of area to perimeter)
        import cv2
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            largest = max(cnts, key=cv2.contourArea)
            area = cv2.contourArea(largest)
            perim = cv2.arcLength(largest, True)
            if perim > 0:
                compactness = (4 * np.pi * area) / (perim ** 2)  # 1.0 = circle, lower = irregular
            else:
                compactness = 0.0
        else:
            compactness = 0.0

        # Fill distance from extremes (prefer 0.15-0.45 range)
        ideal_fill = 0.30
        fill_score = 1.0 - min(1.0, abs(fill - ideal_fill) / 0.30)

        # Weighted quality score
        quality = (
            fill_score * 0.4 +          # 40% weight on reasonable fill
            (1.0 - border_penalty) * 0.3 +  # 30% weight on not touching borders
            compactness * 0.3            # 30% weight on compactness
        )

        return float(np.clip(quality, 0.0, 1.0))

    # Check area validity
    f_gray = area_fraction(gray_mask)
    f_color = area_fraction(color_mask)

    gray_ok = (area_lo <= f_gray <= area_hi)
    color_ok = (area_lo <= f_color <= area_hi)

    # Only one is valid → use that one
    if color_ok and not gray_ok:
        return color_mask, "color"
    if gray_ok and not color_ok:
        return gray_mask, "gray"

    # Both invalid → use gray as fallback
    if not gray_ok and not color_ok:
        return gray_mask, "gray"

    # Both valid → compare quality
    q_gray = mask_quality(gray_mask)
    q_color = mask_quality(color_mask)

    # Decision with configurable tie-breaking
    if abs(q_gray - q_color) < 0.1:  # Within 10% = tie
        return (color_mask, "color") if prefer_color else (gray_mask, "gray")
    elif q_color > q_gray:
        return color_mask, "color"
    else:
        return gray_mask, "gray"


def mask_confidence_mask(mask_u8):
    """
    Very simple mask-quality heuristic:
    - 'ok' if fill is in a reasonable band and the mask doesn't hug too many borders.
    - 'low' otherwise.
    """
    if mask_u8 is None:
        return "low"

    m = (mask_u8 > 0).astype("uint8")
    fill = m.mean()

    touches = {
        "top":    (m[0, :]  > 0).any(),
        "bottom": (m[-1, :] > 0).any(),
        "left":   (m[:, 0]  > 0).any(),
        "right":  (m[:, -1] > 0).any(),
    }
    border_touch = sum(touches.values())

    # Too small, too large, or touching too many edges
    if fill < 0.01 or fill > 0.85 or border_touch >= 3:
        return "low"

    return "ok"

# --- Color/Tonal audit helpers (advisory only) ---

def near_guard_rv(rv, pf, tol=0.02):
    lo, hi = PROFILES[pf]["bands"]["rv"]["guard"]
    return (abs(rv - lo) <= tol) or (abs(rv - hi) <= tol)

def apply_roi_to_L(L_u8, dx_roi):
    """Mirror Δx ROI for luminance reads."""
    if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
        H = L_u8.shape[0]
        return L_u8[: max(1, int(H * float(dx_roi["top"]))), :]
    return L_u8

def tonal_stats_L(L_u8):
    """Return (S_L, eta_L, beta_L) on 8-bit L in [0..255]."""
    import numpy as np, cv2
    # Span (P95 - P5) scaled to [0..1]
    p5, p95 = np.percentile(L_u8, 5), np.percentile(L_u8, 95)
    S_L = float((p95 - p5) / 255.0)

    # Otsu separability η in [0,1]
    hist = cv2.calcHist([L_u8], [0], None, [256], [0, 256]).ravel()
    p = hist / max(1, hist.sum())
    bins = np.arange(256)
    mu_T = (p * bins).sum()
    sigma_T2 = (p * (bins - mu_T) ** 2).sum() + 1e-9
    w0 = np.cumsum(p); w1 = 1.0 - w0
    mu0 = np.cumsum(p * bins) / np.maximum(w0, 1e-9)
    mu1 = (mu_T - np.cumsum(p * bins)) / np.maximum(w1, 1e-9)
    sigma_B2 = (w0 * (mu0 - mu_T) ** 2 + w1 * (mu1 - mu_T) ** 2).max()
    eta_L = float(np.clip(sigma_B2 / sigma_T2, 0.0, 1.0))

    # Bright mass β_L (≥ 0.80)
    beta_L = float((L_u8 >= int(0.80 * 255)).mean())
    return S_L, eta_L, beta_L

In [ ]:

# @title Core primitives
import numpy as np

def centroid_delta_x(gray, cfg, dx_roi=None, mode="hybrid_masked"):
    g = ensure_gray_array(gray)

    # Optional Landscape crop (top fraction)
    if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
        top = float(dx_roi.get("top", 0.60))
        H = g.shape[0]
        g = g[: max(1, int(H * top)), :]

    g = resize_longest(g, cfg["preprocessing"]["longest_side"])
    g = cv2.bilateralFilter(g, d=5, sigmaColor=25, sigmaSpace=25)

    # Foreground mask
    fg = foreground_mask(g, cfg)
    fg_u8 = (fg > 0).astype(np.uint8) * 255

    # Edge map
    e = cv2.Canny(g, 50, 100).astype(np.uint8)

    # --- masked-edges first (hybrid), then fallback to mask centroid ---
    m = None
    if mode in ("edge", "edge_masked", "hybrid_masked"):
        e_use = e if mode == "edge" else cv2.bitwise_and(e, fg_u8)
        m = cv2.moments(e_use)

    if m is None or m["m00"] <= 1e-6:
        m = cv2.moments(fg_u8)
        if m["m00"] <= 1e-6:
            return float("nan")  # ← was 0.0

    cx = m["m10"] / (m["m00"] + 1e-6)
    W  = g.shape[1]
    dx = abs(cx - (W / 2.0)) / (W / 2.0)
    return float(np.clip(dx, 0.0, 1.0))

def void_ratio(gray, cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    mask = foreground_mask(gray, cfg)
    fill = (mask > 0).mean()
    rv = float(np.clip(1.0 - fill, 0.0, 1.0))  # more background = higher void
    return rv

def rho_r(gray, cfg, rho_mask_cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    gray = cv2.bilateralFilter(gray, d=5, sigmaColor=25, sigmaSpace=25)
    fg = foreground_mask(gray, cfg)
    if rho_mask_cfg.get("type") == "subject_halo":
        m = subject_halo_mask(fg, halo_frac=rho_mask_cfg.get("halo_frac", 0.10))
    else:
        m = np.ones_like(fg, dtype=np.uint8)*255
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    energy = np.abs(lap) / 255.0
    sel = energy[m > 0]
    if sel.size == 0:
        return 0.0
    val = float(sel.mean())
    # gentle scale to [0,1]
    return float(np.clip(val * CONFIG.get("rho_scale", 4.0), 0.0, 1.0))

def measure_primitives(path_or_gray, cfg, rho_mask_cfg, dx_roi=None):
    gray = load_image_robust(path_or_gray) if isinstance(path_or_gray, (str, bytes)) else path_or_gray
    dx  = centroid_delta_x(gray, cfg, dx_roi=dx_roi)
    rv  = void_ratio(gray, cfg)
    rho = rho_r(gray, cfg, rho_mask_cfg)
    return {"dx": dx, "rv": rv, "rho": rho}


In [ ]:

# @title Scoring kernel

def band_flag(val, guard):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return "RED"
    lo, hi = guard
    return "OK" if (lo <= val <= hi) else "RED"

def gaussian_score(val, guard):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return 0.0
    lo, hi = guard
    c = 0.5 * (lo + hi)
    sigma = max(1e-6, CONFIG.get("sigma_scale", 0.25) * (hi - lo))
    return float(np.clip(math.exp(-0.5 * ((val - c) / sigma) ** 2), 0.0, 1.0))

def score_with_profile(prims, profile_cfg, cfg):
    bands = profile_cfg["bands"]
    w = profile_cfg["weights"]
    # band flags
    b_dx  = band_flag(prims["dx"],  bands["dx"]["guard"])
    b_rv  = band_flag(prims["rv"],  bands["rv"]["guard"])
    b_rho = band_flag(prims["rho"], bands["rho"]["guard"])

    # soft band scores
    s_dx  = gaussian_score(prims["dx"],  bands["dx"]["guard"])
    s_rv  = gaussian_score(prims["rv"],  bands["rv"]["guard"])
    s_rho = gaussian_score(prims["rho"], bands["rho"]["guard"])

    # weighted geometric mean (epsilon-safe)
    eps = 1e-6
    k = (
        max(eps, s_dx )**w["dx"] *
        max(eps, s_rv )**w["rv"] *
        max(eps, s_rho)**w["rho"]
    ) ** (1.0 / (w["dx"] + w["rv"] + w["rho"]))

    LSI = 100.0 * k
    accepted = (LSI >= cfg["accept"]["gate_100"]) and (b_dx!="RED") and (b_rv!="RED") and (b_rho!="RED")

    return {
        "delta_x": round(prims["dx"], 3),
        "void_ratio": round(prims["rv"], 3),
        "rupture_rho": round(prims["rho"], 3),
        "K_lite": round(k, 3),
        "LSI_lite_100": round(LSI, 1),
        "band_delta_x": b_dx,
        "band_r_v": b_rv,
        "band_rho_r": b_rho,
        "accepted": bool(accepted),
    }


In [ ]:

# @title Reset: clear /content/images
import shutil, os
IMG_DIR = "/content/images"
if os.path.exists(IMG_DIR):
    shutil.rmtree(IMG_DIR)
os.makedirs(IMG_DIR, exist_ok=True)
print("Reset:", IMG_DIR)


In [ ]:

# @title Option A — Classic multiple‑file uploader (preferred)
from google.colab import files
uploaded = files.upload()
upload_paths = []
for name, data in uploaded.items():
    path = f"/content/images/{name}"
    with open(path, "wb") as f:
        f.write(data)
    upload_paths.append(path)
print("Uploaded:", len(upload_paths), "files")


In [ ]:

# @title Option B — Upload a ZIP of images (fallback)
from google.colab import files
import zipfile, io, os, glob
z = files.upload()
assert len(z)==1, "Upload exactly one .zip"
name, bytes_ = next(iter(z.items()))
assert name.lower().endswith(".zip"), "This path expects a .zip file"
with zipfile.ZipFile(io.BytesIO(bytes_), 'r') as zip_ref:
    zip_ref.extractall("/content/images")
upload_paths = sorted([p for p in glob.glob("/content/images/**/*", recursive=True)
                       if os.path.splitext(p)[1].lower() in [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]])
print("Extracted:", len(upload_paths), "images to /content/images")


In [ ]:

# @title Option C — Google Drive folder copy (batch)
from google.colab import drive
import shutil, os, glob
drive.mount('/content/drive')
DRIVE_FOLDER = ""  # @param {type:"string"}
assert DRIVE_FOLDER, "Set DRIVE_FOLDER to a Drive path, e.g. /content/drive/MyDrive/my_images"
ex = [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]
srcs = [p for p in glob.glob(os.path.join(DRIVE_FOLDER, "**/*"), recursive=True)
        if os.path.splitext(p)[1].lower() in ex]
for p in srcs:
    shutil.copy2(p, "/content/images/")
print("Copied:", len(srcs), "images into /content/images")


In [ ]:

# @title Profile selector (widget + fallback)
try:
    import ipywidgets as widgets
    profile_mode_widget = widgets.Dropdown(
        options=["Auto","Figure_Default","MarkMaking_Expressive","Landscape"],
        value="Auto",
        description="profile_mode",
    )
    display(profile_mode_widget)
except Exception as e:
    print("Widget not available; using string fallback. Set profile_mode manually.")
profile_mode = "MarkMaking_Expressive"  # @param ["Auto","Figure_Default","MarkMaking_Expressive","Landscape"]


In [ ]:
import matplotlib.image as mpimg
import glob, os

paths = sorted(glob.glob("/content/images/*"))
n = min(12, len(paths))
if n == 0:
    print("No images in /content/images yet. Use Option A2/A3 or B above.")
else:
    cols = 4; rows = (n + cols - 1)//cols
    plt.figure(figsize=(cols*3, rows*3))
    for i,p in enumerate(paths[:n], 1):
        plt.subplot(rows, cols, i)
        plt.imshow(mpimg.imread(p))
        plt.title(os.path.basename(p)[:30], fontsize=8); plt.axis("off")
    plt.show()

import matplotlib.pyplot as plt

def show_color_row(rgb_u8, gray_u8, fg_gray_mask_u8=None):
    try:
        lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
        L = lab[:, :, 0]
        fg_color_mask = color_mask_via_lab_kmeans(rgb_u8, k=3, morph_kernel=CONFIG["preprocessing"]["morph_kernel"])
        edges_L = cv2.Canny(L, 50, 100)
        edges_L_masked = cv2.bitwise_and(edges_L, edges_L, mask=fg_color_mask)

        plt.figure(figsize=(10, 3))
        plt.subplot(1, 3, 1); plt.imshow(fg_color_mask, cmap="gray"); plt.title("Color FG Mask"); plt.axis("off")
        if fg_gray_mask_u8 is not None:
            plt.subplot(1, 3, 2); plt.imshow(fg_gray_mask_u8, cmap="gray"); plt.title("Gray FG Mask"); plt.axis("off")
        else:
            plt.subplot(1, 3, 2); plt.imshow(gray_u8, cmap="gray"); plt.title("Gray"); plt.axis("off")
        plt.subplot(1, 3, 3); plt.imshow(edges_L_masked, cmap="gray"); plt.title("L-Edges ∩ Color FG"); plt.axis("off")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("[Color row] skipped:", e)

if INCLUDE_COLOR:
    paths = sorted(glob.glob("/content/images/*"))
    if paths:
        g, r = load_image_robust(paths[0])
        show_color_row(r, g)

In [ ]:
# ------------------------------------------------------------
# Run scoring (per-image loop)
# ------------------------------------------------------------
from glob import glob
import os, math
import numpy as np
import pandas as pd
import cv2

# Helper: reason_for_fail (uses scoring kernel outputs)
def _reason_for_fail(row):
    # hard band reds
    if any(row.get(k, "") == "RED" for k in ("band_delta_x", "band_rv", "band_rho_r")):
        return "band_red"

    # LSI gate 100 (if enabled)
    if row.get("LST_lite_100", 1) < CONFIG["accept"].get("gate_100", 0):
        return "lsi_gate"

    return ""

# ------------------------------------------------------------
# Image discovery
# ------------------------------------------------------------
exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff")

# If an uploader cell has already defined upload_paths, reuse it.
if "upload_paths" in globals() and isinstance(upload_paths, list) and len(upload_paths) > 0:
    # Filter to known extensions, just in case
    upload_paths = [
        p for p in upload_paths
        if os.path.splitext(p)[1].lower() in exts
    ]
else:
    # Fallback: scan the Colab directory written by the uploaders
    upload_paths = sorted(
        [
            p
            for p in glob("/content/images/*")
            if os.path.splitext(p)[1].lower() in exts
        ]
    )

print("Found", len(upload_paths), "images")
for p in upload_paths[:5]:
    print("  ", p)

# ------------------------------------------------------------
# Read chosen mode from widget (or fallback string)
# ------------------------------------------------------------
try:
    chosen_mode = profile_mode_widget.value
except NameError:
    chosen_mode = profile_mode  # fallback if widget not present
print("Using profile_mode:", chosen_mode)

rows = []

# ------------------------------------------------------------
# Main per-image loop
# ------------------------------------------------------------
for path in upload_paths:
    # ---- load image ----
    gray, rgb = load_image_robust(path)

    # ---- choose profile ----
    pf = chosen_mode
    if chosen_mode == "Auto":
        quick_dx = centroid_delta_x(gray, CONFIG, dx_roi=None)
        quick_rv = void_ratio(gray, CONFIG)
        pf = auto_profile(quick_dx, quick_rv)

    rho_mask_cfg = PROFILES[pf]["rho_mask"]

    # ---- optional ROI for Δx (Landscape only, if configured) ----
    dx_roi = None
    if pf == "Landscape" and PROFILES[pf].get("dx_roi") == "auto":
        dx_roi = auto_dx_roi(gray) or None

    # ---- core primitives (gray-based) ----
    prims = measure_primitives(gray, CONFIG, rho_mask_cfg, dx_roi=dx_roi)
    dx_gray = float(prims["dx"])
    rv_gray = float(prims["rv"])
    rho_val = float(prims["rho"])   # halo-based rupture metric Note: rupture ALWAYS stays gray-based

    # ---- gray subject mask (foreground) ----
    gray_mask = foreground_mask(gray, CONFIG)

    # --------------------------------------------------------
    # COLOR telemetry + candidate subject mask (non-gating) Compute alternate segmentation (may replace gray)
    # --------------------------------------------------------
    color_mask = None
    dx_color = None
    rv_color = None
    mask_mode = "gray"
    color_status = "not_used"

    if INCLUDE_COLOR and rgb is not None:
        try:
            color_mask, dx_color, rv_color, mask_mode, color_status = color_telemetry(
                gray,
                rgb,
                morph_kernel=CONFIG["preprocessing"]["morph_kernel"],
            )
        except Exception:
            # hard failure in color path → fall back silently to gray
            color_mask, dx_color, rv_color = None, None, None
            mask_mode, color_status = "gray", "error"

    # --------------------------------------------------------
    # Unified subject mask + confidence. Mask selection: color wins if it has better subject isolation
    # --------------------------------------------------------
    if color_mask is not None:
        final_mask, mask_source = choose_subject_mask(
            gray_mask,
            color_mask,
            area_lo=0.03,
            area_hi=0.65,
            prefer_color=True,
        )
    else:
        final_mask, mask_source = gray_mask, "gray"

    mask_confidence = mask_confidence_mask(final_mask)

    # choose Δx / rᵥ based on mask_source + availability
    final_dx, final_rv = dx_gray, rv_gray

    # CRITICAL: If color mask won, its measurements replace gray in scoring
    if mask_source in ("color", "external"):
        if (dx_color is not None) and (rv_color is not None):
            final_dx = float(dx_color)    # ← Used in LSI calculation
            final_rv = float(rv_color)    # ← Used in LSI calculation
        # else: keep gray metrics but still report mask_source, confidence

    # overwrite primitives for scoring
    prims["dx"] = final_dx
    prims["rv"] = final_rv
    prims["rho"] = rho_val

    # --------------------------------------------------------
    # Score with the selected profile
    # --------------------------------------------------------
    row = score_with_profile(prims, PROFILES[pf], CONFIG)
    row.update(
        {
            "name": os.path.basename(path),
            "profile": pf,
        }
    )

    # --------------------------------------------------------
    # COLOR TELEMETRY (non-gating)
    # --------------------------------------------------------
    delta_rv = (
        rv_color - final_rv
        if (rv_color is not None) and not math.isnan(final_rv)
        else float("nan")
    )
    delta_dx = (
        dx_color - final_dx
        if (dx_color is not None) and not math.isnan(final_dx)
        else float("nan")
    )

    color_audit_badge = ""
    if (
        (rv_color is not None)
        and (dx_color is not None)
        and not math.isnan(delta_rv)
        and not math.isnan(delta_dx)
    ):
        # large disagreement between gray vs color subject
        if abs(delta_rv) >= 0.85 or abs(delta_dx) >= 0.20:
            color_audit_badge = "Color audit: subject/mask disagreement"

    row.update(
        {
            "dx_gray": dx_gray,
            "rv_gray": rv_gray,
            "dx_color": dx_color,
            "rv_color": rv_color,
            "mask_mode": mask_mode,
            "color_delta_rv": delta_rv,
            "color_delta_dx": delta_dx,
            "mask_source": mask_source,
            "mask_confidence": mask_confidence,
            "color_audit_badge": color_audit_badge,
        }
    )

    # --------------------------------------------------------
    # TONAL TELEMETRY + AUDIT (non-gating; same ROI as Δx)
    # --------------------------------------------------------
    tonal_status = "none"
    tonal_audit_badge = ""

    try:
        if rgb is not None:
            lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
            L_full = lab[:, :, 0]

            if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
                L_roi = apply_roi_to_L(L_full, dx_roi)
            else:
                L_roi = L_full

            t_mu, t_beta_L = tonal_stats_L(L_roi)
            tonal_status = "ok"
            # simple audit: extremely low variance / crushed range
            if t_beta_L < 0.12:
                tonal_status = "flat"
                tonal_audit_badge = "Tonal audit: possible washout / low contrast"
    except Exception:
        tonal_status = "fail_convert"
        tonal_audit_badge = "Tonal audit: conversion failure"

    # merge badges
    if color_audit_badge and tonal_audit_badge:
        combined_badge = "; ".join([color_audit_badge, tonal_audit_badge])
    else:
        combined_badge = color_audit_badge or tonal_audit_badge

    row.update(
        {
            "tonal_status": tonal_status,
            "tonal_audit_badge": combined_badge,
        }
    )

    # --------------------------------------------------------
    # reason_for_fail + collect row
    # --------------------------------------------------------
    row["reason_for_fail"] = _reason_for_fail(row)
    rows.append(row)

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------
df = pd.DataFrame(rows)

# Final rounding / display
df_out = df.copy()
for col in df_out.columns:
    if df_out[col].dtype.kind in "fc":
        df_out[col] = df_out[col].round(3)

display(df_out)


In [ ]:
# 2) Disable truncation for the session
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
df.head()
display(df)

In [ ]:
# Build 'base' from the scored output for downstream analysis
base = df_out.copy()

# Ensure there is a 'frame' column for TEL / ambiguity logic
if 'frame' not in base.columns:
    if 'name' in base.columns:
        base['frame'] = base['name']          # use the filename
    else:
        base['frame'] = base.index.astype(str)  # fallback: index as label

print("base ready with columns:", base.columns.tolist())
df_out

In [ ]:
# === TEL: Add corridor_90 and cadence_cv to base ===
import numpy as np, pandas as pd, cv2, os

assert 'base' in globals(), "Run scoring first to create `base`"

def _ensure_gray_tel(path):
    """Load and return grayscale uint8."""
    from PIL import Image, ImageOps
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    rgb = np.asarray(img)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def _mask_from_gray_tel(gray_u8, k=5):
    """Simple Otsu mask (dark on light)."""
    blur = cv2.GaussianBlur(gray_u8, (5,5), 0)
    _, bin_ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (255 - bin_).astype(np.uint8)
    K = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, K, 1)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  K, 1)
    return m

def corridor_90_tel(mask_u8):
    """Return (fraction, left_px, right_px) for narrowest 90% lane."""
    cols = mask_u8.sum(axis=0).astype(float)
    total = cols.sum()
    if total <= 0: return 1.0, 0, mask_u8.shape[1]-1

    target = 0.90 * total
    L, s = 0, 0.0
    best = (mask_u8.shape[1], 0, mask_u8.shape[1]-1)

    for R in range(len(cols)):
        s += cols[R]
        while s - cols[L] >= target:
            s -= cols[L]; L += 1
        if s >= target:
            w = R - L + 1
            if w < best[0]: best = (w, L, R)

    return float(best[0]/mask_u8.shape[1]), int(best[1]), int(best[2])

def cadence_cv_tel(mask_u8, L, R):
    """CV of vertical blank runs inside lane [L, R]."""
    if R <= L: return 0.0
    lane = mask_u8[:, L:R+1]
    if lane.size == 0: return 0.0

    # blank = rows with no ink
    blank_rows = (lane.sum(axis=1) == 0)
    runs, run_len = [], 0
    for is_blank in blank_rows:
        if is_blank:
            run_len += 1
        elif run_len > 0:
            runs.append(run_len); run_len = 0
    if run_len > 0: runs.append(run_len)

    if len(runs) < 2: return 0.0
    runs = np.array(runs, float)
    mu = runs.mean()
    return float(runs.std(ddof=1) / mu) if mu > 0 else 0.0

# Process each frame
c90_list, mL_list, mR_list, cad_list = [], [], [], []
frames = base['frame'].astype(str).tolist()

for fname in frames:
    path = fname if os.path.isabs(fname) else f"/content/images/{fname}"

    try:
        g = _ensure_gray_tel(path)
        m = _mask_from_gray_tel(g, k=5)
        c90, L, R = corridor_90_tel(m)
        cad = cadence_cv_tel(m, L, R)
    except:
        c90, L, R, cad = np.nan, np.nan, np.nan, np.nan

    c90_list.append(c90)
    mL_list.append(L)
    mR_list.append(R)
    cad_list.append(cad)

base['corridor_90'] = c90_list
base['c90_L'] = mL_list
base['c90_R'] = mR_list
base['cadence_cv'] = cad_list

print("✓ TEL metrics added:", base[['corridor_90','cadence_cv']].describe())

In [ ]:
import re

def parse_ambiguity(s):
    """Extract iteration number from patterns like 'A1 Iter', 'B15 Iter', etc."""
    # Try pattern: [Letter][Number] Iter
    m = re.search(r'([A-Z])(\d+)\s+Iter', str(s), re.I)
    if m:
        return float(m.group(2))

    # Fallback: look for any standalone number
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan

# Recompute
base['ambiguity'] = base['frame'].apply(parse_ambiguity)

print(f"✓ Ambiguity parsed: {base['ambiguity'].nunique()} unique values")
print(f"  Range: [{base['ambiguity'].min():.2f}, {base['ambiguity'].max():.2f}]")
print(f"  Missing: {base['ambiguity'].isna().sum()} rows")

In [ ]:
# === Fix: Use table order as iteration/ambiguity (filename-agnostic) ===
base['iteration'] = range(1, len(base) + 1)
base['ambiguity'] = base['iteration'].astype(float)
print(f"✓ Auto-assigned iterations 1-{len(base)} based on upload order")

# === Ambiguity Sweep Readout ===
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# Check we have what we need
required = ['delta_x', 'void_ratio', 'rupture_rho', 'corridor_90', 'cadence_cv', 'ambiguity']
missing = [c for c in required if c not in base.columns]
if missing:
    raise RuntimeError(f"Missing columns: {missing}. Run scoring + TEL + ambiguity cells first.")

# Clean data
df = base[required].copy()
df = df.replace([np.inf, -np.inf], np.nan)

# Helper: median [IQR]
def miqr(s):
    s = s.dropna()
    if len(s) == 0: return "n/a"
    q1, q3 = s.quantile([0.25, 0.75])
    return f"{s.median():.3f} [{q1:.3f}–{q3:.3f}]"

# 1) Per-level summaries
print("\n=== Medians [IQR] by Ambiguity Level ===")
levels = sorted(df['ambiguity'].dropna().unique())
for lvl in levels:
    sub = df[df['ambiguity'] == lvl]
    print(f"\nAmb = {lvl:.2f} (n={len(sub)})")
    print(f"  Δx:          {miqr(sub['delta_x'])}")
    print(f"  r_v:         {miqr(sub['void_ratio'])}")
    print(f"  ρ_r:         {miqr(sub['rupture_rho'])}")
    print(f"  corridor_90: {miqr(sub['corridor_90'])}")
    print(f"  cadence_cv:  {miqr(sub['cadence_cv'])}")

# 2) Correlations
print("\n=== Spearman ρ with Ambiguity ===")
for name, col in [
    ("ρ_r (packing)",  'rupture_rho'),
    ("cadence_cv",     'cadence_cv'),
    ("r_v (void)",     'void_ratio'),
    ("Δx",             'delta_x'),
    ("corridor_90",    'corridor_90'),
]:
    mask = df['ambiguity'].notna() & df[col].notna()
    if mask.sum() < 3:
        print(f"{name:15s}: insufficient data")
        continue
    rho, p = spearmanr(df.loc[mask, 'ambiguity'], df.loc[mask, col])
    sig = "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{name:15s}: ρ = {rho:+.3f} (p = {p:.3f}) {sig}")

# 3) Plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ρ_r vs ambiguity
axes[0].scatter(df['ambiguity'], df['rupture_rho'], s=32, alpha=0.7)
axes[0].set_xlabel('Ambiguity'); axes[0].set_ylabel('ρ_r (packing)')
axes[0].set_title('Packing vs Ambiguity')
axes[0].grid(alpha=0.3)

# cadence_cv vs ambiguity
axes[1].scatter(df['ambiguity'], df['cadence_cv'], s=32, alpha=0.7, color='C1')
axes[1].set_xlabel('Ambiguity'); axes[1].set_ylabel('cadence_cv')
axes[1].set_title('Cadence vs Ambiguity')
axes[1].grid(alpha=0.3)

# corridor_90 vs r_v (colored by ambiguity)
sc = axes[2].scatter(df['corridor_90'], df['void_ratio'],
                     c=df['ambiguity'], s=32, alpha=0.7, cmap='viridis')
axes[2].set_xlabel('corridor_90'); axes[2].set_ylabel('r_v (void)')
axes[2].set_title('Lane vs Void (colored by ambiguity)')
axes[2].grid(alpha=0.3)
plt.colorbar(sc, ax=axes[2], label='ambiguity')

plt.tight_layout()
plt.show()

# 4) Summary verdict
print("\n" + "="*60)
print("PREDICTION CHECK:")
print("  ρ_r & cadence_cv ↑ with ambiguity → surface creativity")
print("  Δx & corridor_90 ~ flat           → spatial defaults persist")
print("  If TRUE: style-space ≠ composition-space (your thesis!)")
print("="*60)

In [ ]:
print("\nQuick check:")
print(base[['frame', 'delta_x', 'void_ratio', 'rupture_rho',
            'corridor_90', 'cadence_cv', 'ambiguity']])

In [ ]:
# === Recursion Analysis (iteration trajectory) ===
import re
import matplotlib.pyplot as plt

# === Use table order as iteration (filename-agnostic) ===
base['iteration'] = range(1, len(base) + 1)

# Plot metrics over iterations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].plot(base['iteration'], base['delta_x'], 'o-')
axes[0,0].set_ylabel('Δx (off-center)'); axes[0,0].set_title('Centroid Drift')
axes[0,0].axhline(0.05, color='r', linestyle='--', alpha=0.3, label='centered')
axes[0,0].set_xlabel('Iteration')

axes[0,1].plot(base['iteration'], base['void_ratio'], 'o-')
axes[0,1].set_ylabel('r_v (void)'); axes[0,1].set_title('Void Ratio')
axes[0,1].set_xlabel('Iteration')

axes[1,0].plot(base['iteration'], base['rupture_rho'], 'o-')
axes[1,0].set_ylabel('ρ_r (packing)'); axes[1,0].set_title('Mark Energy')
axes[1,0].set_xlabel('Iteration')

axes[1,1].plot(base['iteration'], base['corridor_90'], 'o-')
axes[1,1].set_ylabel('corridor_90'); axes[1,1].set_title('Lane Width')
axes[1,1].set_xlabel('Iteration')

for ax in axes.flat:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Trajectory Check ===")
print("Δx trend:", "improving ↑" if base['delta_x'].iloc[-1] > base['delta_x'].iloc[0] else "regressing ↓")
print("r_v trend:", "improving ↑" if base['void_ratio'].iloc[-1] > base['void_ratio'].iloc[0] else "regressing ↓")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(base['void_ratio'], base['rupture_rho'],
           c=base['delta_x'], cmap='viridis', s=100, alpha=0.7)
plt.colorbar(label='Δx (displacement)')
plt.xlabel('r_v (void ratio)')
plt.ylabel('ρ_r (packing)')
plt.title('Compositional Space (2D projection)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print(f"dx range: {base['delta_x'].min():.3f} - {base['delta_x'].max():.3f}")
print(f"rv range: {base['void_ratio'].min():.3f} - {base['void_ratio'].max():.3f}")
print(f"rho range: {base['rupture_rho'].min():.3f} - {base['rupture_rho'].max():.3f}")

In [ ]:
# === Basin plot with exposed X,Y coordinates ===
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt

# Cluster in compositional space
X = base[['delta_x', 'void_ratio', 'rupture_rho']].values
X = X[np.all(np.isfinite(X), axis=1)]

n_samples = X.shape[0]
if n_samples == 0:
    raise ValueError("No finite samples available for basin clustering.")

# Use up to 4 basins, but never more than we have samples for
n_basins = min(4, n_samples)

kmeans = KMeans(n_clusters=n_basins, random_state=42).fit(X)
base['basin'] = kmeans.labels_

print(f"Using {n_basins} basin(s) for {n_samples} sample(s).")
print("Basin sizes:", base['basin'].value_counts().sort_index())

# Compute barycentric coordinates
def to_barycentric(dx, rv, rho):
    """Convert (Δx, r_v, ρ_r) to barycentric (x, y) coordinates."""

    # Use 5th-95th percentile instead of min-max to ignore outliers
    dx_min, dx_max = base['delta_x'].quantile([0.05, 0.95])
    rv_min, rv_max = base['void_ratio'].quantile([0.05, 0.95])
    rho_min, rho_max = base['rupture_rho'].quantile([0.05, 0.95])

    # Clip to range, then normalize
    dx_clipped = np.clip(dx, dx_min, dx_max)
    rv_clipped = np.clip(rv, rv_min, rv_max)
    rho_clipped = np.clip(rho, rho_min, rho_max)

    dx_n = (dx_clipped - dx_min) / (dx_max - dx_min + 1e-9)
    rv_n = (rv_clipped - rv_min) / (rv_max - rv_min + 1e-9)
    rho_n = (rho_clipped - rho_min) / (rho_max - rho_min + 1e-9)

    # Row-normalize to barycentric weights
    total = dx_n + rv_n + rho_n + 1e-9
    w_dx = dx_n / total
    w_rv = rv_n / total
    w_rho = rho_n / total

    # Triangle vertices
    A = np.array([0.0, 0.0])
    B = np.array([1.0, 0.0])
    C = np.array([0.5, np.sqrt(3)/2])

    x = w_dx * A[0] + w_rv * B[0] + w_rho * C[0]
    y = w_dx * A[1] + w_rv * B[1] + w_rho * C[1]

    return x, y, w_dx, w_rv, w_rho

# Calculate coordinates for all points
coords = base.apply(lambda row: to_barycentric(
    row['delta_x'], row['void_ratio'], row['rupture_rho']
), axis=1)

base['bary_x'] = [c[0] for c in coords]
base['bary_y'] = [c[1] for c in coords]
base['weight_dx'] = [c[2] for c in coords]
base['weight_rv'] = [c[3] for c in coords]
base['weight_rho'] = [c[4] for c in coords]

# Plot with coordinates
plt.figure(figsize=(8, 8))

# Triangle frame
A = np.array([0.0, 0.0])
B = np.array([1.0, 0.0])
C = np.array([0.5, np.sqrt(3)/2])
tri_x = [A[0], B[0], C[0], A[0]]
tri_y = [A[1], B[1], C[1], A[1]]
plt.plot(tri_x, tri_y, 'k-', lw=2)

# Vertex labels (bigger and clearer)
offset = 0.08
plt.text(A[0]-offset, A[1]-offset, 'Δx\n(placement)',
         ha='right', va='top', fontsize=11, weight='bold')
plt.text(B[0]+offset, B[1]-offset, 'r_v\n(void)',
         ha='left', va='top', fontsize=11, weight='bold')
plt.text(C[0], C[1]+offset, 'ρ_r\n(packing)',
         ha='center', va='bottom', fontsize=11, weight='bold')

# Plot points colored by basin
for basin in range(n_basins):
    mask = base['basin'] == basin
    plt.scatter(base.loc[mask, 'bary_x'],
                base.loc[mask, 'bary_y'],
                label=f'Basin {basin}', s=50, alpha=0.7)

# Add coordinate labels for each point (optional - can be overwhelming)
# Uncomment if you want to see all coordinates:
# for idx, row in base.iterrows():
#     plt.text(row['bary_x'], row['bary_y'],
#              f"({row['bary_x']:.2f},{row['bary_y']:.2f})",
#              fontsize=6, ha='center')

# Add grid lines (tertiary grid)
for w in [0.2, 0.4, 0.6, 0.8]:
    # Lines parallel to each edge
    # Parallel to AB (bottom edge)
    plt.plot([w*C[0], w*C[0] + (1-w)*B[0]],
             [w*C[1], w*C[1] + (1-w)*B[1]],
             'k-', lw=0.3, alpha=0.3)
    # Parallel to BC
    plt.plot([w*A[0] + (1-w)*B[0], w*A[0] + (1-w)*C[0]],
             [w*A[1] + (1-w)*B[1], w*A[1] + (1-w)*C[1]],
             'k-', lw=0.3, alpha=0.3)
    # Parallel to CA
    plt.plot([w*B[0] + (1-w)*A[0], w*B[0] + (1-w)*C[0]],
             [w*B[1] + (1-w)*A[1], w*B[1] + (1-w)*C[1]],
             'k-', lw=0.3, alpha=0.3)

plt.legend(loc='upper right', frameon=True)
plt.title("Compositional Basins (with coordinates)", fontsize=14)
plt.axis('equal')
plt.axis('off')
plt.tight_layout()
plt.show()

# Print coordinate table for reference
print("\n=== BARYCENTRIC COORDINATES BY BASIN: ===")
for basin in range(n_basins):
    sub = base[base['basin'] == basin].sort_values('LSI_lite_100',
                                                   ascending=False).head(3)
    print(f"\nBasin {basin} (top 3 by LSI):")
    for idx, row in sub.iterrows():
        print(
            f"  {row['frame']!s:>12}: "
            f"x={row['bary_x']:.3f}, y={row['bary_y']:.3f} "
            f" | Δx={row['weight_dx']:.2f}, r_v={row['weight_rv']:.2f}, "
            f"ρ_r={row['weight_rho']:.2f}"
        )

In [ ]:
# Label specific points (e.g., Basin 1 exemplars)
basin1_best = base[base['basin'] == 1].sort_values('LSI_lite_100', ascending=False).iloc[0]

plt.figure(figsize=(8, 8))

# Triangle frame
A = np.array([0.0, 0.0])
B = np.array([1.0, 0.0])
C = np.array([0.5, np.sqrt(3)/2])
tri_x = [A[0], B[0], C[0], A[0]]
tri_y = [A[1], B[1], C[1], A[1]]
plt.plot(tri_x, tri_y, 'k-', lw=2)

# Vertex labels
offset = 0.08
plt.text(A[0]-offset, A[1]-offset, 'Δx\n(placement)',
         ha='right', va='top', fontsize=11, weight='bold')
plt.text(B[0]+offset, B[1]-offset, 'r_v\n(void)',
         ha='left', va='top', fontsize=11, weight='bold')
plt.text(C[0], C[1]+offset, 'ρ_r\n(packing)',
         ha='center', va='bottom', fontsize=11, weight='bold')

# Plot points colored by basin
for basin in range(4):
    mask = base['basin'] == basin
    plt.scatter(base.loc[mask, 'bary_x'],
                base.loc[mask, 'bary_y'],
                label=f'Basin {basin}', s=50, alpha=0.7)

# Annotate Basin 1 best
plt.annotate(f"Basin 1 best\n({basin1_best['bary_x']:.2f}, {basin1_best['bary_y']:.2f})",
             xy=(basin1_best['bary_x'], basin1_best['bary_y']),
             xytext=(basin1_best['bary_x']+0.1, basin1_best['bary_y']+0.1),
             arrowprops=dict(arrowstyle='->', lw=1),
             fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))

# Add titles
plt.xlabel('Barycentric X (weighted composition)', fontsize=12)
plt.ylabel('Barycentric Y (weighted composition)', fontsize=12)
plt.title('Compositional Basins: Barycentric Triangle (Δx, r_v, ρ_r)', fontsize=14, weight='bold')

plt.legend(loc='upper right', frameon=True)
plt.axis('equal')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Basin Quality ===")
for basin in range(4):
    sub = base[base['basin'] == basin]
    pass_rate = sub['accepted'].mean()
    print(f"Basin {basin}: {pass_rate*100:.0f}% pass rate")

In [ ]:
# === Basin Assignment Table ===
import pandas as pd

# Create a summary table with basin assignments
basin_summary = base[['frame', 'basin', 'LSI_lite_100', 'accepted',
                      'delta_x', 'void_ratio', 'rupture_rho']].copy()

# Sort by basin, then by LSI score within each basin
basin_summary = basin_summary.sort_values(['basin', 'LSI_lite_100'],
                                          ascending=[True, False])

# Add a readable status
basin_summary['status'] = basin_summary['accepted'].map({True: '✓ Pass', False: '✗ Fail'})

# Round numeric columns for readability
for col in ['LSI_lite_100', 'delta_x', 'void_ratio', 'rupture_rho']:
    basin_summary[col] = basin_summary[col].round(2)

# Display the full table
print("="*80)
print("BASIN ASSIGNMENTS (sorted by basin, then LSI score)")
print("="*80)
pd.set_option('display.max_rows', None)
display(basin_summary[['frame', 'basin', 'status', 'LSI_lite_100',
                       'delta_x', 'void_ratio', 'rupture_rho']])

# Summary statistics per basin
print("\n" + "="*80)
print("BASIN SUMMARY STATISTICS")
print("="*80)
for basin in range(4):
    subset = base[base['basin'] == basin]
    print(f"\nBasin {basin}: {len(subset)} images ({subset['accepted'].sum()} pass, {(~subset['accepted']).sum()} fail)")
    print(f"  LSI range: {subset['LSI_lite_100'].min():.1f} - {subset['LSI_lite_100'].max():.1f}")
    print(f"  Median Δx: {subset['delta_x'].median():.3f}")
    print(f"  Median r_v: {subset['void_ratio'].median():.3f}")
    print(f"  Median ρ_r: {subset['rupture_rho'].median():.3f}")

In [ ]:
print("\n=== Basin Compositional Profiles ===")
for basin in range(4):
    sub = base[base['basin'] == basin]
    print(f"\n{'='*50}")
    print(f"BASIN {basin} (n={len(sub)}, pass rate={sub['accepted'].mean()*100:.0f}%)")
    print(f"{'='*50}")
    print(f"Δx (off-center):  {sub['delta_x'].median():.3f} [{sub['delta_x'].quantile(0.25):.3f}–{sub['delta_x'].quantile(0.75):.3f}]")
    print(f"r_v (void):       {sub['void_ratio'].median():.3f} [{sub['void_ratio'].quantile(0.25):.3f}–{sub['void_ratio'].quantile(0.75):.3f}]")
    print(f"ρ_r (packing):    {sub['rupture_rho'].median():.3f} [{sub['rupture_rho'].quantile(0.25):.3f}–{sub['rupture_rho'].quantile(0.75):.3f}]")
    print(f"corridor_90:      {sub['corridor_90'].median():.3f}")
    print(f"LSI score:        {sub['LSI_lite_100'].median():.1f}")

    # Which images are in this basin?
    print(f"Examples: {', '.join(sub['frame'].head(3).astype(str))}")

In [ ]:
# === Generate Basin Exemplar Figure ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

def basin_exemplar_figure(df, n_basins=4):
    """Create 4-panel figure showing one exemplar per basin."""

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes = axes.flatten()

    for basin in range(n_basins):
        sub = df[df['basin'] == basin].sort_values('LSI_lite_100', ascending=False)

        if len(sub) == 0:
            continue

        # Get best-scoring image from this basin
        exemplar = sub.iloc[0]
        img_path = f"/content/images/{exemplar['frame']}"

        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            axes[basin].imshow(img)

            # Annotation
            title = f"Basin {basin} (n={len(sub)}, pass={sub['accepted'].mean()*100:.0f}%)\n"
            title += f"Δx={exemplar['delta_x']:.2f}, r_v={exemplar['void_ratio']:.2f}, ρ_r={exemplar['rupture_rho']:.2f}"
            axes[basin].set_title(title, fontsize=10)
        else:
            axes[basin].text(0.5, 0.5, f"Basin {basin}\nImage not found",
                           ha='center', va='center')

        axes[basin].axis('off')

    plt.suptitle("Compositional Basins: Representative Examples", fontsize=14, y=0.98)
    plt.tight_layout()
    plt.savefig("/content/basin_exemplars.png", dpi=200, bbox_inches='tight')
    plt.show()
    print("✅ Saved to /content/basin_exemplars.png")

basin_exemplar_figure(base)

In [ ]:
# === 0910E TEL ADD-ONS (corridor_90, cadence_cv, margins) — advisory only ===
import numpy as np, pandas as pd, math, os
import matplotlib.pyplot as plt

# Optional deps (only used if we need to recompute masks from image paths)
try:
    import cv2
    from skimage.morphology import opening, closing, disk
    from skimage.filters import threshold_otsu
except Exception as _e:
    cv2 = None

# 0) Auto-discover the ledger table without renames
candidate_tables = [
    obj for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
]

ledger = None
for tbl in candidate_tables:
    if {'void_ratio','rupture_rho'}.issubset(tbl.columns):
        ledger = tbl.copy()
        break

tbl = ledger.copy()

# 1) Alias TEL_rv_from_mask (robust void) if not present
if "TEL_rv_from_mask" not in tbl.columns and "void_ratio" in tbl.columns:
    tbl["TEL_rv_from_mask"] = pd.to_numeric(tbl["void_ratio"], errors="coerce")

# 2) Helpers for TEL metrics, optionally recomputing mask if needed
def _load_gray(path):
    if cv2 is None or not isinstance(path, str) or not os.path.exists(path):
        return None
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return im

def _mask_from_gray(gray, k=5, dark_on_light=True):
    # Otsu + polarity + small open/close to stabilize corridor reads
    t = threshold_otsu(gray)
    ink = (gray < t) if dark_on_light else (gray > t)
    m = opening(ink.astype(np.uint8)*255, footprint=disk(k))
    m = closing(m, footprint=disk(k))
    return (m > 0).astype(np.uint8)

def _corridor_90(mask):
    # returns corridor width fraction, and left/right margins (fractions)
    H, W = mask.shape
    col = mask.sum(axis=0).astype(np.float64)
    total = col.sum()
    if total <= 0:
        return np.nan, np.nan, np.nan
    target = 0.90 * total
    # two-pointer minimal contiguous window
    s = 0.0
    L = 0
    best = (0, W+1)
    for R in range(W):
        s += col[R]
        while s - col[L] >= target and L < R:
            s -= col[L]; L += 1
        if s >= target and (R - L + 1) < (best[1] - best[0] + 1):
            best = (L, R)
    left, right = best
    width = (right - left + 1)
    corridor_frac = float(np.clip(width / W, 0, 1))
    margin_L = float(np.clip(left / W, 0, 1))
    margin_R = float(np.clip((W - right - 1) / W, 0, 1))
    return corridor_frac, margin_L, margin_R

def _cadence_cv(mask, left_frac, right_frac):
    # cadence within the corridor: CV of vertical blank-run lengths per column (median over corridor)
    if np.isnan(left_frac) or np.isnan(right_frac):
        return np.nan
    H, W = mask.shape
    L = max(0, min(W-1, int(round(left_frac * W))))
    R = max(0, min(W-1, int(round((1.0 - right_frac) * W)) - 1))
    if R < L:
        L, R = R, L
    runs_cv = []
    for x in range(L, R+1):
        col = mask[:, x].astype(bool)
        # blank runs = consecutive zeros
        lengths = []
        run = 0
        for v in ~col:
            if v:
                run += 1
            elif run > 0:
                lengths.append(run); run = 0
        if run > 0:
            lengths.append(run)
        if len(lengths) >= 2:
            arr = np.array(lengths, dtype=np.float64)
            mu = arr.mean()
            sig = arr.std(ddof=1) if len(arr) > 1 else 0.0
            if mu > 0:
                runs_cv.append(sig / mu)
    if len(runs_cv) == 0:
        return 0.0  # solid fill or perfectly regular blanks
    return float(np.median(runs_cv))

# 3) Compute corridor_90, margins, cadence_cv if missing and we can recover an ink mask
need_corridor = "corridor_90" not in tbl.columns
need_cadence  = "cadence_cv"  not in tbl.columns
need_margins  = not {"c90_margin_L","c90_margin_R"}.issubset(set(tbl.columns))

if need_corridor or need_cadence or need_margins:
    # try to find an image path column; common names:
    path_col = next((c for c in ["frame_path","path","frame","image","img_path"] if c in tbl.columns), None)
    if path_col is None and (need_corridor or need_cadence):
        print("No path column found; skipping recomputation of corridor/cadence. (Existing values will be used if present.)")
    else:
        c90_list, mL_list, mR_list, cad_list = [], [], [], []
        for _, row in tbl.iterrows():
            if not need_corridor and not need_cadence and not need_margins:
                break
            mask = None
            # If your pipeline stored a binary mask per-row, you can read it here; otherwise rebuild from image:
            p = str(row[path_col]) if path_col else None
            gray = _load_gray(p) if p else None
            if gray is not None:
                mask = _mask_from_gray(gray, k=5, dark_on_light=True)
            if mask is None:
                c90_list.append(np.nan); mL_list.append(np.nan); mR_list.append(np.nan); cad_list.append(np.nan)
                continue
            c90, mL, mR = _corridor_90(mask)
            cad = _cadence_cv(mask, mL, mR)
            c90_list.append(c90); mL_list.append(mL); mR_list.append(mR); cad_list.append(cad)
        if need_corridor and "corridor_90" not in tbl.columns:
            tbl["corridor_90"] = c90_list
        if need_margins:
            if "c90_margin_L" not in tbl.columns: tbl["c90_margin_L"] = mL_list
            if "c90_margin_R" not in tbl.columns: tbl["c90_margin_R"] = mR_list
        if need_cadence and "cadence_cv" not in tbl.columns:
            tbl["cadence_cv"] = cad_list

# 4) Margin skew (optional, robust to zeros)
if "margin_skew_log2" not in tbl.columns and {"c90_margin_L","c90_margin_R"}.issubset(set(tbl.columns)):
    tbl["margin_skew_log2"] = (
        np.log2(pd.to_numeric(tbl["c90_margin_L"], errors="coerce") + 1e-9)
        - np.log2(pd.to_numeric(tbl["c90_margin_R"], errors="coerce") + 1e-9)
    ).abs()

# 5) Print a compact TEL summary (medians with IQR)
def _med_iqr(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce").dropna()
    if len(s) == 0: return "n/a"
    q1, q3 = np.percentile(s, [25,75])
    return f"{np.median(s):.3f} [{q1:.3f}–{q3:.3f}]"

print("\n--- TEL summary (medians [IQR]) ---")
print("r_v (TEL)        :", _med_iqr(tbl.get("TEL_rv_from_mask", np.nan)))
print("ρ_r (rupture)    :", _med_iqr(tbl.get("rupture_rho", np.nan)))
print("corridor_90      :", _med_iqr(tbl.get("corridor_90", np.nan)))
print("cadence_cv       :", _med_iqr(tbl.get("cadence_cv", np.nan)))
if {"c90_margin_L","c90_margin_R"}.issubset(set(tbl.columns)):
    print("margin_L_to_R    :", _med_iqr(tbl["c90_margin_L"] / (pd.to_numeric(tbl["c90_margin_R"], errors="coerce")+1e-9)))

# 6) Write back into the original ledger object (no renames)
ledger.loc[tbl.index, tbl.columns] = tbl[tbl.columns]
print("\nTEL add-ons applied (advisory only). Gate logic unchanged.")

In [ ]:
# === 0910E STATS + PLOT (advisory; no gating) ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt, re, math, random
from scipy.stats import spearmanr

# 0) Use base as the TEL stats table (no auto-discovery)
tbl = base.copy()
print("Using table: base")

# 1) Make sure the TEL series exist
rv  = pd.to_numeric(tbl.get("TEL_rv_from_mask", tbl.get("void_ratio")), errors="coerce")
rho = pd.to_numeric(tbl.get("rupture_rho"), errors="coerce")

c90_raw = tbl.get("corridor_90")
if c90_raw is None:
    print("No 'corridor_90' column found – skipping TEL correlation block.")
    # You can either bail out of the cell here, or just skip the perm_p_spearman section.
else:
    c90 = pd.to_numeric(c90_raw, errors="coerce")

    # === 1) Spearman between corridor_90 and r_v (TEL) with permutation p and bootstrap CI ===
    def perm_p_spearman(x, y, n_perm=5000, seed=42):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float); y = np.asarray(y, float)
        mask = np.isfinite(x) & np.isfinite(y)
        x, y = x[mask], y[mask]
        if len(x) < 5:
            return np.nan, np.nan, (np.nan, np.nan)
        rho_obs, _ = spearmanr(x, y)

        # permutation p
        count = 0
        for _ in range(n_perm):
            yp = rng.permutation(y)
            rho_p, _ = spearmanr(x, yp)
            if abs(rho_p) >= abs(rho_obs):
                count += 1
        p = (count + 1) / (n_perm + 1)

        # bootstrap CI
        rng = np.random.default_rng(seed + 1)
        reps = 3000
        boots = []
        n = len(x)
        for _ in range(reps):
            idx = rng.integers(0, n, n)
            r, _ = spearmanr(x[idx], y[idx])
            if np.isfinite(r):
                boots.append(r)
        lo, hi = np.percentile(boots, [2.5, 97.5]) if len(boots) > 10 else (np.nan, np.nan)
        return rho_obs, p, (lo, hi)

    rho_xy, p_xy, (lo_xy, hi_xy) = perm_p_spearman(c90, rv)
    print("\n--- Correlation (TEL) ---")
    print(
        f"Spearman ρ(corridor_90, r_v TEL) = {rho_xy:.3f} "
        f"(perm p = {p_xy:.3f}; 95% bootstrap CI [{lo_xy:.3f}, {hi_xy:.3f}])"
    )

# 2) Optional Cliff's delta if a grouping exists (e.g., 'group' column or 'Early/Late')
def cliffs_delta(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0: return np.nan
    # dominance count
    tot = 0
    for ai in a:
        tot += np.sum(ai > b) - np.sum(ai < b)
    return tot / (len(a)*len(b))

group_col = None
for cand in ["group","cohort","is_late","phase"]:
    if cand in tbl.columns:
        group_col = cand; break

if group_col is not None:
    g = tbl[group_col]
    if g.nunique() == 2:
        g1, g2 = list(g.dropna().unique())
        a = rv[g == g1]; b = rv[g == g2]
        # bootstrap CI
        rng = np.random.default_rng(123)
        reps = 3000
        deltas = []
        for _ in range(reps):
            aa = a.sample(len(a), replace=True, random_state=rng.integers(0, 1<<31))
            bb = b.sample(len(b), replace=True, random_state=rng.integers(0, 1<<31))
            deltas.append(cliffs_delta(aa.values, bb.values))
        d_obs = cliffs_delta(a.values, b.values)
        lo, hi = np.percentile(deltas, [2.5, 97.5]) if len(deltas) > 10 else (np.nan, np.nan)
        print(f"\nCliff's δ({g2} − {g1}) on r_v TEL = {d_obs:.3f} (95% CI [{lo:.3f}, {hi:.3f}])")
    else:
        print("\nCliff's δ: skipped (group has ≠2 levels).")
else:
    print("\nCliff's δ: skipped (no binary grouping column found).")

# 3) Key scatter: corridor_90 vs r_v (TEL) with per-iteration labels if available
labels = None
for cand in ["iter","iteration","frame_idx"]:
    if cand in tbl.columns:
        labels = pd.to_numeric(tbl[cand], errors="coerce").astype("Int64")
        break
if labels is None:
    labels = pd.Series(range(1, len(tbl)+1))

plt.figure(figsize=(5.5, 4.5))
plt.scatter(c90, rv, s=36)
for xi, yi, lab in zip(c90, rv, labels):
    if np.isfinite(xi) and np.isfinite(yi):
        plt.text(xi, yi, str(int(lab)), fontsize=8, ha='center', va='center')
plt.xlabel("corridor_90 (90% lane / page width)")
plt.ylabel("r_v (TEL, robust void)")
plt.title("TEL: lane width vs void")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 4) Results snippet (copy into your doc)
def _med_iqr(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce").dropna()
    if len(s)==0: return "n/a"
    q1, q3 = np.percentile(s, [25, 75])
    return f"{np.median(s):.3f} [{q1:.3f}–{q3:.3f}]"

print("\n--- Results box (copy-paste) ---")
print(f"Pass rate (gate): {int(tbl.get('accepted', pd.Series(dtype=bool)).sum())} / {len(tbl)}")
print("r_v (TEL)   median [IQR]:", _med_iqr(rv))
print("ρ_r         median [IQR]:", _med_iqr(rho))
print("corridor_90 median [IQR]:", _med_iqr(c90))
if "cadence_cv" in tbl.columns:
    print("cadence_cv  median [IQR]:", _med_iqr(tbl["cadence_cv"]))
print("Parity (single vs batch): aligned")  # ops note, unchanged

# === table (numbers behind the dots) ===

cols = ['frame', 'corridor_90']

# Use TEL_rv_from_mask if present, otherwise fall back to void_ratio
if 'TEL_rv_from_mask' in base.columns:
    rv_col_name = 'TEL_rv_from_mask'
elif 'void_ratio' in base.columns:
    rv_col_name = 'void_ratio'
else:
    raise KeyError("No r_v column found (TEL_rv_from_mask or void_ratio).")

cols.append(rv_col_name)

# (optional) include packing as a fourth column if available
if 'rupture_rho' in base.columns:
    cols.append('rupture_rho')

tbl = base[cols].copy()

# add iteration index as first column
tbl.insert(0, 'iter', np.arange(1, len(tbl) + 1))
if 'rupture_rho' in base.columns:
    tbl['rupture_rho'] = pd.to_numeric(base['rupture_rho'], errors='coerce')

# tidy formatting
for c in ['corridor_90','TEL_rv_from_mask','rupture_rho']:
    if c in tbl.columns:
        tbl[c] = pd.to_numeric(tbl[c], errors='coerce').round(3)

# show all rows and keep order by iter
pd.set_option('display.max_rows', None)
display(tbl)

# (optional) save for copy/paste or sharing
# tbl.to_csv('/content/scroll_TEL_table.csv', index=False)

In [ ]:
# @title Plots
import matplotlib.pyplot as plt

# 0) Ensure we are plotting from the right dataframe
if 'base' in globals() and not base.empty:
    df_plot = base.copy()
elif 'df_out' in globals() and not df_out.empty:
    df_plot = df_out.copy()
elif 'df' in globals() and not df.empty:
    df_plot = df.copy()
else:
    print("No dataframe to plot. Run scoring first.")
    df_plot = None

if df_plot is not None:
    xs = list(range(1, len(df_plot) + 1))

    # --- Core primitives over iterations ---
    plt.figure(figsize=(6, 4))
    plt.plot(xs, df_plot['delta_x'],    label='Δx')
    plt.plot(xs, df_plot['void_ratio'], label='r_v')
    plt.plot(xs, df_plot['rupture_rho'],label='ρ_r')
    plt.xlabel("Iteration")
    plt.ylabel("value (0–1)")
    plt.title("Primitives over iterations")
    plt.legend()
    plt.show()

    # --- LSI-lite over iterations ---
    lsi_candidates = ['LSI_lite_100', 'LSI_Lite_100', 'LSI_LITE_100']
    metric_col = None
    for c in lsi_candidates:
        if c in df_plot.columns:
            metric_col = c
            break

    if metric_col is not None:
        plt.figure(figsize=(6, 4))
        plt.plot(xs, df_plot[metric_col], label=metric_col)
        plt.xlabel("Iteration")
        plt.ylabel("LSI-lite (0–100)")
        plt.title("LSI-lite over iterations")
        plt.legend()
        plt.show()
    else:
        print("No LSI-lite column found in df_plot; columns are:")
        print(list(df_plot.columns))

    # --- Audit rate (Color or Tonal) ---
    mask_color = df_plot["color_audit_badge"].astype(str).str.len() > 0
    mask_tonal = (
        df_plot["tonal_audit_badge"].astype(str).str.len() > 0
        if "tonal_audit_badge" in df_plot.columns
        else False
    )
    mask = (mask_color | mask_tonal)
    audit_rate = (mask.sum() / len(df_plot)) if len(df_plot) else 0.0
    print(f"Audit rate: {audit_rate*100:.1f}% (Color or Tonal)")

In [ ]:

# @title Export (CSV + simple HTML)
import pandas as pd, os
CSV_PATH = "/content/LSI_lite_results.csv"
HTML_PATH = "/content/LSI_lite_report.html"
if 'df' in globals() and not df.empty:
    df.to_csv(CSV_PATH, index=False)
    html = "<h2>LSI_lite results</h2>" + df.to_html(index=False)
    with open(HTML_PATH, "w") as f:
        f.write(html)
    print("Saved:", CSV_PATH, "and", HTML_PATH)
else:
    print("Nothing to export; run scoring first.")


In [ ]:
# === Barycentric table + triangle for (Δx, rᵥ, ρᵣ) — self-contained, no schema changes ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt, re

# 1) Auto-pick the active ledger DataFrame (must already be in memory)
ledger_df = None
for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        cols = [str(c).lower() for c in obj.columns]
        if any("delta_x" in c for c in cols) and any(("void" in c or "r_v" in c) for c in cols) and any("rho" in c for c in cols):
            ledger_df = obj
            print(f"Using table: {name}")
            break
assert ledger_df is not None, "No DataFrame with Δx/void/ρ columns found. Run your ledger build cell first."

df = ledger_df.copy()

# 2) Column pickers (fuzzy, keep your current names)
def pick(keys):
    for c in df.columns:
        cl = str(c).lower()
        if all(k in cl for k in keys):
            return c
    # looser fallback
    for c in df.columns:
        cl = str(c).lower()
        if any(k in cl for k in keys):
            return c
    raise KeyError(keys)

dx_col   = pick(["delta_x"])          # e.g., "delta_x"
rv_col   = pick(["void"])             # e.g., "void_ratio" or "r_v"
rho_col  = pick(["rho"])              # e.g., "rupture_rho"
titlecol = next((c for c in df.columns if "title" in str(c).lower() or "frame" in str(c).lower()), None)
datecol  = next((c for c in df.columns if "date"  in str(c).lower()), None)

# 3) Numeric series (no renames)
for c in [dx_col, rv_col, rho_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 4) Feature-wise min–max to [0,1], then row-normalize to true barycentric weights (sum=1)
eps = 1e-12
def f_norm(x):
    x = x.astype(float)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x) + eps)

DX  = f_norm(df[dx_col].to_numpy())
RV  = f_norm(df[rv_col].to_numpy())
RHO = f_norm(df[rho_col].to_numpy())
W   = np.stack([DX, RV, RHO], axis=1)
W   = np.nan_to_num(W, nan=0.0)
row_sums = W.sum(axis=1, keepdims=True) + eps
BW = W / row_sums   # barycentric weights per row: (wΔx, w_rv, w_rho)

# 5) Convert barycentric weights to 2D triangle coordinates
A   = np.array([0.0, 0.0])                 # Δx vertex
Bv  = np.array([1.0, 0.0])                 # rᵥ vertex
C   = np.array([0.5, np.sqrt(3)/2])        # ρᵣ vertex
XY  = BW[:,[0]]*A + BW[:,[1]]*Bv + BW[:,[2]]*C
X, Y = XY[:,0], XY[:,1]

# 6) Optional year parsing for reference (not required)
def start_year(s):
    m = re.findall(r"(\d{4})", str(s))
    return int(m[0]) if m else np.nan
years = df[datecol].map(start_year) if datecol else pd.Series([np.nan]*len(df))

# 7) Assemble the barycentric table you can sort/export later
bary_df = pd.DataFrame({
    "iter": np.arange(1, len(df)+1),
    "w_delta_x": BW[:,0],
    "w_void_rv": BW[:,1],
    "w_packing_rho": BW[:,2],
    "bary_x": X,
    "bary_y": Y,
})
if titlecol: bary_df.insert(1, "title", df[titlecol].astype(str))
if datecol:  bary_df.insert(2, "date",  df[datecol].astype(str))
if datecol:  bary_df.insert(3, "StartYear", years)

# 8) Show the table (you can: bary_df.sort_values("StartYear") later)
display(bary_df)

# 9) Draw the triangle frame + labeled vertices + numbered points
plt.figure(figsize=(6,6))
# triangle frame
tri_x = [A[0], Bv[0], C[0], A[0]]
tri_y = [A[1], Bv[1], C[1], A[1]]
plt.plot(tri_x, tri_y, lw=1.0, color="gray")

# vertex labels
plt.text(A[0]-0.03, A[1]-0.03, "Δx (placement)", ha="right", va="top", fontsize=9)
plt.text(Bv[0]+0.03, Bv[1]-0.03, "rᵥ (void)",     ha="left",  va="top", fontsize=9)
plt.text(C[0], C[1]+0.04,          "ρᵣ (packing)", ha="center", va="bottom", fontsize=9)

# points
plt.scatter(X, Y, s=36, edgecolors="k")
for i, (x, y) in enumerate(zip(X, Y), start=1):
    plt.text(x, y, str(i), fontsize=8, ha="center", va="center")

plt.title("Barycentric triangle — (Δx, rᵥ, ρᵣ)")
plt.axis("equal"); plt.axis("off")
plt.show()

In [ ]:
plt.tight_layout()
plt.savefig("/content/barycentric_triangle.png", dpi=200)
plt.show()

In [ ]:
from scipy.spatial import ConvexHull
pts = np.c_[bary_df["bary_x"].values, bary_df["bary_y"].values]
mask = np.all(np.isfinite(pts), axis=1)
area = np.nan
if mask.sum() >= 3:
    area = ConvexHull(pts[mask]).area  # or .volume for 2D area in some SciPy versions
print(f"Hull area (strategy spread): {area:.3f}" if np.isfinite(area) else "Hull area: n/a")

bary_df.to_csv("/content/barycentric_table.csv", index=False)

NOT PART OF CANON VERSION BELOW

In [ ]:
# === cadence_cv vs rupture_rho (materials rhythm vs energy) ===
import matplotlib.pyplot as plt
x = pd.to_numeric(base['rupture_rho'], errors='coerce')
y = pd.to_numeric(base['cadence_cv'], errors='coerce')
m = x.notna() & y.notna()

plt.figure(figsize=(4.8,4))
plt.scatter(x[m], y[m], s=32)
for i,(xi,yi) in enumerate(zip(x[m], y[m]), start=1):
    plt.text(xi+0.005, yi+0.005, str(i), fontsize=8)
plt.xlabel("ρ_r (rupture_rho)")
plt.ylabel("cadence_cv (TEL)")
plt.title("Cadence vs mark energy")
plt.grid(alpha=.2); plt.show()

In [ ]:
# === TEL: convex hull area in (delta_x, void_ratio, rupture_rho) ===
import numpy as np

def hull_area_2d(pts):
    # Andrew's monotone chain on 2D points
    pts = sorted(set(map(tuple, pts)))
    if len(pts) < 3: return 0.0
    def cross(o,a,b): return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lower = []
    for p in pts:
        while len(lower)>=2 and cross(lower[-2], lower[-1], p) <= 0: lower.pop()
        lower.append(p)
    upper = []
    for p in reversed(pts):
        while len(upper)>=2 and cross(upper[-2], upper[-1], p) <= 0: upper.pop()
        upper.append(p)
    hull = lower[:-1] + upper[:-1]
    # polygon area
    area = 0.0
    for i in range(len(hull)):
        x1,y1 = hull[i]
        x2,y2 = hull[(i+1)%len(hull)]
        area += x1*y2 - x2*y1
    return abs(area)/2.0

pts = np.c_[pd.to_numeric(base['delta_x'], errors='coerce'),
            pd.to_numeric(base['void_ratio'], errors='coerce'),
            pd.to_numeric(base['rupture_rho'], errors='coerce')]
# area in a 2D projection that matters most here: (r_v, ρ_r)
A = hull_area_2d(pts[:,[1,2]])
print(f"Hull area (r_v vs ρ_r): {A:.3f}  (bigger = more spread of strategies)")

In [ ]:
# === TEL results box (use on AI or Artists; assumes `base` exists) ===
import numpy as np, pandas as pd
from scipy.stats import spearmanr

tbl = base.copy()

def med_iqr(s):
    s = pd.to_numeric(s, errors='coerce').dropna()
    return f"{np.median(s):.3f} [{np.percentile(s,25):.3f}–{np.percentile(s,75):.3f}]"

# prefer TEL r_v from the robust mask if present
rv_col = 'TEL_rv_from_mask' if 'TEL_rv_from_mask' in tbl.columns else 'void_ratio'

print(f"Pass rate: {int((tbl.get('accepted', False)==True).sum())} / {len(tbl)}")
print(f"r_v (void, TEL)   median [IQR]: {med_iqr(tbl[rv_col])}")
print(f"ρ_r (rupture_rho) median [IQR]: {med_iqr(tbl['rupture_rho'])}")
if 'corridor_90' in tbl.columns:
    print(f"corridor_90       median [IQR]: {med_iqr(tbl['corridor_90'])}")
if 'margin_L_to_R' in tbl.columns:
    print(f"margin_L_to_R     median [IQR]: {med_iqr(tbl['margin_L_to_R'])}")

# optional: correlation between lane width and void (TEL)
if 'corridor_90' in tbl.columns:
    x = pd.to_numeric(tbl['corridor_90'], errors='coerce')
    y = pd.to_numeric(tbl[rv_col],        errors='coerce')
    mask = x.notna() & y.notna()
    rho, p = spearmanr(x[mask], y[mask])
    print(f"Spearman ρ(corridor_90, r_v TEL): {rho:.3f}  (p={p:.3f})")

print("Parity (single vs batch): aligned")

In [ ]:
# === Fix: Use table order as iteration/ambiguity (filename-agnostic) ===
base['iteration'] = range(1, len(base) + 1)
base['ambiguity'] = base['iteration'].astype(float)
print(f"✓ Auto-assigned iterations 1-{len(base)} based on upload order")

# Plot the journey
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(base['iteration'], base['LSI_lite_100'], 'o-', label='LSI score')
ax.axhline(55, color='r', ls='--', alpha=0.3, label='gate')
# Overlay basin membership
for basin in range(4):
    mask = base['basin'] == basin
    ax.scatter(base.loc[mask, 'iteration'],
               base.loc[mask, 'LSI_lite_100'],
               label=f'Basin {basin}', s=100, alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('LSI score')
ax.legend()
plt.show()

In [ ]:
# === TRAJECTORY COHERENCE ANALYSIS ===
# Add this cell after your existing LSI-lite scoring
# Analyzes how smoothly the system explores compositional space

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

# Assumes 'base' DataFrame exists with columns: iteration, delta_x, void_ratio, rupture_rho, basin

# ============================================================
# 1. TRAJECTORY JUMP ANALYSIS
# ============================================================

def trajectory_jumps(df, metrics=['delta_x', 'void_ratio', 'rupture_rho']):
    """
    Calculate Euclidean distance between consecutive iterations.
    Large jumps = compositional discontinuities (context shifts?)
    """
    jumps = []
    iterations = []

    df_sorted = df.sort_values('iteration').copy()

    for i in range(1, len(df_sorted)):
        prev = df_sorted.iloc[i-1][metrics].values.astype(float)
        curr = df_sorted.iloc[i][metrics].values.astype(float)

        if np.any(np.isnan(prev)) or np.any(np.isnan(curr)):
            jumps.append(np.nan)
        else:
            jump = np.linalg.norm(curr - prev)
            jumps.append(jump)

        iterations.append(df_sorted.iloc[i]['iteration'])

    return pd.DataFrame({
        'iteration': iterations,
        'jump_size': jumps
    })

jumps_df = trajectory_jumps(base)

# Statistics
median_jump = jumps_df['jump_size'].median()
q75 = jumps_df['jump_size'].quantile(0.75)
large_jumps = jumps_df[jumps_df['jump_size'] > q75 * 1.5]

print("\n" + "="*60)
print("TRAJECTORY COHERENCE ANALYSIS")
print("="*60)
print(f"\nMedian jump size: {median_jump:.3f}")
print(f"75th percentile: {q75:.3f}")
print(f"\nLarge jumps (>1.5 × Q75):")
if len(large_jumps) > 0:
    for _, row in large_jumps.iterrows():
        print(f"  Iteration {int(row['iteration'])}: jump = {row['jump_size']:.3f}")
    print(f"\n→ {len(large_jumps)} potential context window shifts detected")
else:
    print("  None detected (smooth trajectory)")

# ============================================================
# 2. BASIN TRANSITION MATRIX
# ============================================================

def basin_transitions(df):
    """
    Track how system moves between compositional basins.
    Shows whether it finds good strategies and sticks with them.
    """
    df_sorted = df.sort_values('iteration').copy()

    n_basins = df_sorted['basin'].nunique()
    matrix = np.zeros((n_basins, n_basins), dtype=int)

    for i in range(1, len(df_sorted)):
        from_basin = int(df_sorted.iloc[i-1]['basin'])
        to_basin = int(df_sorted.iloc[i]['basin'])
        matrix[from_basin, to_basin] += 1

    return matrix, n_basins

trans_matrix, n_basins = basin_transitions(base)

print("\n" + "="*60)
print("BASIN TRANSITION MATRIX")
print("="*60)
print("(rows = from, columns = to)\n")

# Print matrix
header = "     " + "".join([f"B{i:2d}  " for i in range(n_basins)])
print(header)
for i in range(n_basins):
    row_str = f"B{i}  "
    for j in range(n_basins):
        row_str += f"{trans_matrix[i,j]:4d} "
    print(row_str)

# Diagonal dominance = system stays in same basin (exploiting)
# Off-diagonal = jumping between strategies (exploring)
diagonal_sum = np.trace(trans_matrix)
total_transitions = trans_matrix.sum()
stability = diagonal_sum / max(total_transitions, 1)

print(f"\nStability (stay in same basin): {stability:.1%}")
print(f"Exploration (switch basins): {1-stability:.1%}")

# Which basin is "sticky"?
for i in range(n_basins):
    if trans_matrix[i,i] > 0:
        stay_rate = trans_matrix[i,i] / max(trans_matrix[i,:].sum(), 1)
        print(f"  Basin {i}: {stay_rate:.1%} stay rate")

# ============================================================
# 3. ROLLING HULL AREA (Exploration vs Exploitation)
# ============================================================

def rolling_hull_area(df, window=5, metrics=['delta_x', 'void_ratio', 'rupture_rho']):
    """
    Calculate convex hull area in rolling windows.
    Expanding = exploring new territory
    Stable/shrinking = exploiting known strategies
    """
    df_sorted = df.sort_values('iteration').copy()

    areas = []
    iterations = []

    for i in range(window, len(df_sorted) + 1):
        window_data = df_sorted.iloc[i-window:i]
        points = window_data[metrics].values.astype(float)

        # Remove any rows with NaN
        points = points[~np.any(np.isnan(points), axis=1)]

        if len(points) >= 4:  # Need at least 4 points for 3D hull
            try:
                hull = ConvexHull(points)
                areas.append(hull.volume)  # In 3D, 'volume' is actually volume
                iterations.append(window_data.iloc[-1]['iteration'])
            except:
                areas.append(np.nan)
                iterations.append(window_data.iloc[-1]['iteration'])
        else:
            areas.append(np.nan)
            iterations.append(window_data.iloc[-1]['iteration'])

    return pd.DataFrame({
        'iteration': iterations,
        'hull_area': areas
    })

hull_df = rolling_hull_area(base, window=5)

# Trend: expanding or contracting?
if len(hull_df) > 1:
    first_half = hull_df['hull_area'].iloc[:len(hull_df)//2].median()
    second_half = hull_df['hull_area'].iloc[len(hull_df)//2:].median()

    print("\n" + "="*60)
    print("EXPLORATION DYNAMICS (Rolling Hull Area)")
    print("="*60)
    print(f"First half median: {first_half:.4f}")
    print(f"Second half median: {second_half:.4f}")

    if second_half > first_half * 1.2:
        print("→ EXPANDING: System is exploring new territory")
    elif second_half < first_half * 0.8:
        print("→ CONTRACTING: System is exploiting known strategies")
    else:
        print("→ STABLE: System maintaining consistent exploration range")

# ============================================================
# 4. PLOTS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Jump sizes over iterations
ax = axes[0, 0]
ax.plot(jumps_df['iteration'], jumps_df['jump_size'], 'o-', markersize=4)
ax.axhline(median_jump, color='green', linestyle='--', alpha=0.5, label=f'Median ({median_jump:.3f})')
ax.axhline(q75 * 1.5, color='red', linestyle='--', alpha=0.5, label='Large jump threshold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Jump Size (Euclidean distance)')
ax.set_title('Trajectory Jumps (potential context shifts)')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Rolling hull area
ax = axes[0, 1]
ax.plot(hull_df['iteration'], hull_df['hull_area'], 'o-', markersize=4, color='purple')
ax.set_xlabel('Iteration')
ax.set_ylabel('Hull Volume (3D compositional space)')
ax.set_title('Exploration Range (5-iteration rolling window)')
ax.grid(alpha=0.3)

# Plot 3: Basin transitions as heatmap
ax = axes[1, 0]
im = ax.imshow(trans_matrix, cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(n_basins))
ax.set_yticks(range(n_basins))
ax.set_xticklabels([f'B{i}' for i in range(n_basins)])
ax.set_yticklabels([f'B{i}' for i in range(n_basins)])
ax.set_xlabel('To Basin')
ax.set_ylabel('From Basin')
ax.set_title('Basin Transition Matrix')

# Add text annotations
for i in range(n_basins):
    for j in range(n_basins):
        text = ax.text(j, i, trans_matrix[i, j],
                      ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, ax=ax)

# Plot 4: 3D trajectory with jump annotations
ax = axes[1, 1]
df_sorted = base.sort_values('iteration')
ax.plot(df_sorted['delta_x'], df_sorted['void_ratio'], 'o-', markersize=4, alpha=0.6)

# Mark large jumps
if len(large_jumps) > 0:
    for _, jump_row in large_jumps.iterrows():
        iter_idx = int(jump_row['iteration'])
        row = df_sorted[df_sorted['iteration'] == iter_idx].iloc[0]
        ax.plot(row['delta_x'], row['void_ratio'], 'r*', markersize=12,
                label='Large jump' if _ == large_jumps.index[0] else '')

ax.set_xlabel('Δx (off-center)')
ax.set_ylabel('r_v (void)')
ax.set_title('Compositional Trajectory (2D projection)')
ax.grid(alpha=0.3)
if len(large_jumps) > 0:
    ax.legend()

plt.tight_layout()
plt.savefig('/content/trajectory_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("INTERPRETATION GUIDE")
print("="*60)
print("""
SMOOTH TRAJECTORY + HIGH STABILITY:
  → Artist context is working well
  → System explores within constraints, then converges

JUMPY TRAJECTORY + LOW STABILITY:
  → Context shifts are disrupting exploration
  → System can't exploit successful strategies

EXPANDING HULL + LOW STABILITY:
  → Healthy exploration phase
  → System hasn't found optimal strategy yet

CONTRACTING HULL + HIGH STABILITY:
  → Exploitation phase
  → System found a working strategy and is refining it
""")

# ============================================================
# 5. EXPORT RESULTS
# ============================================================

# Combine all metrics for export
trajectory_summary = pd.DataFrame({
    'median_jump': [median_jump],
    'stability_rate': [stability],
    'exploration_rate': [1 - stability],
    'large_jumps_detected': [len(large_jumps)],
    'first_half_hull': [first_half if len(hull_df) > 1 else np.nan],
    'second_half_hull': [second_half if len(hull_df) > 1 else np.nan]
})

trajectory_summary.to_csv('/content/trajectory_summary.csv', index=False)
jumps_df.to_csv('/content/trajectory_jumps.csv', index=False)
hull_df.to_csv('/content/rolling_hull.csv', index=False)

print("\n✓ Saved trajectory_analysis.png")
print("✓ Saved trajectory_summary.csv")
print("✓ Saved trajectory_jumps.csv")
print("✓ Saved rolling_hull.csv")

In [ ]:
# === RHA@K (Resampled Hull Area at fixed K) for (rᵥ, ρᵣ) ===
# Drop this in AFTER your scoring table exists as a DataFrame (e.g., `df`).
# You can change `DF_NAME` below if your variable is named differently.

import numpy as np, pandas as pd, cv2

# ----- Config -----
DF_NAME      = 'df'     # change to your DataFrame variable name if needed (e.g., 'results')
COL_RV       = 'void_ratio'
COL_RHO      = 'rupture_rho'
K            = 64       # resample size (use the largest K that all cohorts can meet)
B            = 500      # number of resamples for the bootstrap distribution
SEED         = 0        # reproducible resampling
PCT_LOW, PCT_HIGH = 2.5, 97.5  # CI percentiles

# ----- Utilities -----
def _convex_hull_area(points_xy: np.ndarray) -> float:
    """
    points_xy: (n, 2) float32/float64 array in [0,1]^2
    returns convex hull area via OpenCV (shoelace under the hood).
    """
    if points_xy.shape[0] < 3:
        return 0.0
    pts = points_xy.astype(np.float32).reshape(-1, 1, 2)                    # (n,1,2) for cv2
    hull = cv2.convexHull(pts, returnPoints=True)                           # (h,1,2)
    area = float(cv2.contourArea(hull))                                     # area in normalized units
    return max(area, 0.0)

def _hull_vertices(points_xy: np.ndarray) -> np.ndarray:
    """Return hull vertex coordinates (h,2) for eigen-ratio calculation."""
    if points_xy.shape[0] < 3:
        return points_xy
    pts = points_xy.astype(np.float32).reshape(-1, 1, 2)
    hull = cv2.convexHull(pts, returnPoints=True)                           # (h,1,2)
    return hull.reshape(-1, 2).astype(np.float64)

def _eigen_ratio_from_vertices(verts: np.ndarray) -> float:
    """
    PCA on hull vertices; eigen-ratio = major_axis / minor_axis (>=1).
    If degenerate or <2 verts, returns 1.0.
    """
    if verts.shape[0] < 2:
        return 1.0
    X = verts - verts.mean(axis=0, keepdims=True)
    # 2x2 covariance SVD → singular values ~ std along principal axes
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    if S.shape[0] < 2 or S[1] <= 1e-12:
        return 1.0
    return float(S[0] / S[1])

def RHA_at_K(points_xy: np.ndarray, K: int, B: int = 500, seed: int = 0):
    """
    Resampled Hull Area at fixed K with CI + eigen-ratio of the full-cohort hull.
    Returns: dict with median, low/high CI, full_hull_area, eigen_ratio
    """
    n = points_xy.shape[0]
    assert n >= K, f"Not enough points (n={n}) for K={K}. Lower K or add data."
    rng = np.random.default_rng(seed)
    areas = np.empty(B, dtype=np.float64)
    idx = np.arange(n)

    # bootstrap (without replacement) at fixed size K
    for i in range(B):
        samp = rng.choice(idx, size=K, replace=False)
        areas[i] = _convex_hull_area(points_xy[samp])

    areas.sort()
    median = float(np.percentile(areas, 50))
    lo     = float(np.percentile(areas, PCT_LOW))
    hi     = float(np.percentile(areas, PCT_HIGH))

    # shape diagnostics on the full cohort hull
    full_area = _convex_hull_area(points_xy)
    eigen_ratio = _eigen_ratio_from_vertices(_hull_vertices(points_xy))

    return {
        "RHA@K_median": median,
        "RHA@K_CI_low": lo,
        "RHA@K_CI_high": hi,
        "Hull_full": full_area,
        "Hull_eigen_ratio": eigen_ratio,
        "K": int(K),
        "B": int(B)
    }

# ----- Collect points from your DataFrame -----
# Try to fetch your results DataFrame by name; fall back to searching globals.
_g = globals()
df_candidates = []
if DF_NAME in _g and isinstance(_g[DF_NAME], pd.DataFrame):
    df_candidates.append(_g[DF_NAME])
else:
    # Heuristic: pick the first DataFrame in globals that has the required columns
    for v in _g.values():
        if isinstance(v, pd.DataFrame) and {COL_RV, COL_RHO}.issubset(v.columns):
            df_candidates.append(v)

assert df_candidates, f"Could not find a DataFrame '{DF_NAME}' with columns {COL_RV},{COL_RHO}."
df_used = df_candidates[0].copy()

# Optionally filter to accepted frames only (uncomment next line if desired)
# df_used = df_used[df_used.get('accepted', True) == True]

points = df_used[[COL_RV, COL_RHO]].to_numpy(dtype=np.float64)
# Sanity clamp to [0,1]^2 if upstream code drifted
points = np.clip(points, 0.0, 1.0)

# If K is larger than available n, auto-shrink K to n
K_eff = min(K, points.shape[0])
if K_eff < 3:
    raise ValueError(f"Need at least 3 points to form a hull (have {points.shape[0]}).")

result = RHA_at_K(points, K=K_eff, B=B, seed=SEED)

print(
    f"RHA@{result['K']}: {result['RHA@K_median']:.3f} "
    f"[{result['RHA@K_CI_low']:.3f}–{result['RHA@K_CI_high']:.3f}]  |  "
    f"Hull(full)={result['Hull_full']:.3f}  |  eigen-ratio={result['Hull_eigen_ratio']:.2f}"
)

# If you also want Stability-Weighted Hull (SWH) on the RHA@K median:
# (requires pass-rate and mean safety margin in your df; plug your own fields here)
# pass_rate = float(df_used.get('accepted', pd.Series([True]*len(df_used))).mean())
# mean_margin = float(df_used.get('mean_safety_margin', pd.Series([np.nan]*len(df_used))).fillna(0.0).mean())
# SWH = result['RHA@K_median'] * pass_rate * mean_margin
# print(f"SWH (on RHA@{result['K']} median) = {SWH:.3f}   [pass={pass_rate:.2f}, margin={mean_margin:.3f}]")

In [ ]:
# --- RHA@K + optional SWH (read-only; no global mutations) ---
import numpy as np, pandas as pd, cv2

# ======= USER KNOBS (edit here only) =======
DF_CANDIDATE_NAMES = ["df", "results", "table", "cohort"]  # will pick the first that exists
COL_RV, COL_RHO = "void_ratio", "rupture_rho"              # adjust if your column names differ
K_TARGET = 24                                              # pick K < n to get a real CI
B = 500                                                    # resamples
SEED = 0                                                   # reproducibility

# Optional: if you DON'T already have a safety margin column, you can define safe bands here
# and we'll compute a temporary mean safety margin (no columns are written back).
USE_MANUAL_BANDS = True
RV_SAFE = (0.10, 0.90)    # <-- fill with your rᵥ safe band if USE_MANUAL_BANDS=True
RHO_SAFE = (0.06, 0.70)   # <-- fill with your ρᵣ safe band if USE_MANUAL_BANDS=True
ACCEPTED_COL = "accepted" # if present, used to compute pass rate

# ======= helpers =======
def _convex_hull_area(points: np.ndarray) -> float:
    if points.shape[0] < 3: return 0.0
    pts = points.astype(np.float32).reshape(-1,1,2)
    hull = cv2.convexHull(pts, returnPoints=True)
    return float(max(cv2.contourArea(hull), 0.0))

def _hull_vertices(points: np.ndarray) -> np.ndarray:
    if points.shape[0] < 3: return points
    pts = points.astype(np.float32).reshape(-1,1,2)
    hull = cv2.convexHull(pts, returnPoints=True).reshape(-1,2).astype(np.float64)
    return hull

def _eigen_ratio(verts: np.ndarray) -> float:
    if verts.shape[0] < 2: return 1.0
    X = verts - verts.mean(axis=0, keepdims=True)
    _, S, _ = np.linalg.svd(X, full_matrices=False)
    return float(S[0]/S[1]) if S.shape[0] > 1 and S[1] > 1e-12 else 1.0

def _find_df():
    g = globals()
    for name in DF_CANDIDATE_NAMES:
        if name in g and isinstance(g[name], pd.DataFrame):
            df = g[name]
            if {COL_RV, COL_RHO}.issubset(df.columns):
                return df, name
    # fallback: scan all globals
    for k,v in g.items():
        if isinstance(v, pd.DataFrame) and {COL_RV, COL_RHO}.issubset(v.columns):
            return v, k
    raise ValueError(f"No DataFrame with columns {COL_RV},{COL_RHO} found in globals.")

def _mean_safety_margin(df):
    # 1) use existing column if present
    for cand in ["safety_margin","margin","mean_safety_margin"]:
        if cand in df.columns and np.issubdtype(df[cand].dtype, np.number):
            return float(df[cand].mean())
    # 2) compute temporary from manual bands (no writes)
    if USE_MANUAL_BANDS:
        rv_lo, rv_hi = RV_SAFE
        rho_lo, rho_hi = RHO_SAFE
        def norm_margin(x, lo, hi):
            w = max(hi-lo, 1e-9)
            if x<lo or x>hi: return 0.0
            return min(x-lo, hi-x)/w
        m = [
            min(norm_margin(rv,rv_lo,rv_hi), norm_margin(rho,rho_lo,rho_hi))
            for rv,rho in zip(df[COL_RV].values, df[COL_RHO].values)
        ]
        return float(np.mean(m))
    # 3) otherwise, unknown
    return None

# ======= main (read-only) =======
df_src, df_name = _find_df()
points = df_src[[COL_RV, COL_RHO]].to_numpy(dtype=np.float64)
points = np.clip(points, 0.0, 1.0)  # normalized space guard
n = points.shape[0]
if n < 3:
    raise ValueError(f"Need ≥3 points, found n={n} in {df_name}.")

K = min(int(K_TARGET), n)  # if K_TARGET >= n, we’ll still run (CI will collapse)
rng = np.random.default_rng(SEED)
idx = np.arange(n)

areas = np.empty(B, dtype=np.float64)
for i in range(B):
    samp = rng.choice(idx, size=K, replace=False)
    areas[i] = _convex_hull_area(points[samp])

areas.sort()
med = float(np.percentile(areas, 50))
lo  = float(np.percentile(areas, 2.5))
hi  = float(np.percentile(areas, 97.5))

full_area   = _convex_hull_area(points)
eigr        = _eigen_ratio(_hull_vertices(points))

# Pass rate (read if present; else assume all accepted)
if ACCEPTED_COL in df_src.columns:
    pass_rate = float(df_src[ACCEPTED_COL].mean())
else:
    pass_rate = 1.0

mean_margin = _mean_safety_margin(df_src)
SWH = med * pass_rate * mean_margin if mean_margin is not None else None

print(f"[{df_name}] RHA@{K}: {med:.3f} [{lo:.3f}–{hi:.3f}]  |  Hull(full)={full_area:.3f}  |  eigen-ratio={eigr:.2f}")
print(f"PassRate={pass_rate:.2f}" + (f"  |  MeanSafetyMargin={mean_margin:.3f}  |  SWH={SWH:.3f}" if SWH is not None else "  |  MeanSafetyMargin=? (set USE_MANUAL_BANDS=True or provide a column)"))

In [ ]:
# --- AI Default Proximity Index (ADPI) — read-only ranking ---

import numpy as np
import pandas as pd

# Core column mapping
COLS = dict(
    dx="delta_x",
    rv="void_ratio",
    rho="rupture_rho",
    frame="frame",
    iter="iteration",
    accepted="accepted",
)

# Safe bands for r_v and ρ_r (tune from your profile)
RV_SAFE  = (0.55, 0.65)   # example: mid-band for void ratio
RHO_SAFE = (0.20, 0.70)   # example: non-RED band for rupture_rho


def _find_df():
    """Prefer base → df_out → df as the source table."""
    if "base" in globals() and isinstance(base, pd.DataFrame) and not base.empty:
        return base.copy()
    if "df_out" in globals() and isinstance(df_out, pd.DataFrame) and not df_out.empty:
        return df_out.copy()
    if "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
        return df.copy()
    raise ValueError("No suitable DataFrame found (base / df_out / df).")


def ai_default_index(df):
    """Compute a simple 'how default-like' score based on r_v / ρ_r vs safe bands."""
    # Ensure numeric
    for key in ("dx", "rv", "rho"):
        col = COLS[key]
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in df.")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ensure we have a frame label column
    frame_col = COLS["frame"]
    if frame_col not in df.columns:
        df[frame_col] = df.index.astype(str)

    # Ensure we have an iteration column (1..N) for plotting/ranking
    iter_col = COLS["iter"]
    if iter_col not in df.columns:
        df[iter_col] = np.arange(1, len(df) + 1)

    # Ensure accepted flag exists (default True)
    acc_col = COLS["accepted"]
    if acc_col not in df.columns:
        df[acc_col] = True

    dx  = df[COLS["dx"]].to_numpy(dtype=float)
    rv  = df[COLS["rv"]].to_numpy(dtype=float)
    rho = df[COLS["rho"]].to_numpy(dtype=float)

    # Normalize each primitive to a 0–1 "how default-like" distance from the safe band
    rv_lo, rv_hi = RV_SAFE
    rho_lo, rho_hi = RHO_SAFE

    # distance from safe band edges (0 = inside band, 1 = far outside)
    rv_d = np.where(rv < rv_lo, (rv_lo - rv) / (rv_hi - rv_lo),
                    np.where(rv > rv_hi, (rv - rv_hi) / (rv_hi - rv_lo), 0.0))
    rho_d = np.where(rho < rho_lo, (rho_lo - rho) / (rho_hi - rho_lo),
                     np.where(rho > rho_hi, (rho - rho_hi) / (rho_hi - rho_lo), 0.0))

    # combine distances: smaller = more default-like / centered
    # (you can tweak weights; here equal weight)
    d_raw = 0.5 * rv_d + 0.5 * rho_d

    # invert so 1.0 = very default, 0.0 = far from default
    adpi = 1.0 - np.clip(d_raw, 0.0, 1.0)

    # Build compact output table
    out = df[[frame_col, iter_col, COLS["dx"], COLS["rv"], COLS["rho"], acc_col]].copy()
    out["ADPI"] = adpi
    return out.sort_values("ADPI", ascending=False)


# Run it
df_adpi = ai_default_index(_find_df())
df_adpi.head(15)

In [ ]:
# --- Edge Continuity (CLAHE=OFF) : one-cell drop-in ---------------------------
# Requirements: OpenCV (cv2), numpy, pandas, scikit-image
# df must already exist and include a "frame" column with paths or filenames.

import os, math, sys, numpy as np, pandas as pd
from pathlib import Path

import cv2
from skimage.morphology import skeletonize
from scipy import ndimage as ndi

# ------------------------- CONFIG (edit if needed) ----------------------------
IMAGE_ROOTS = [
    ".",                        # try current dir first
    "./images",                 # common subdir
    "./frames",                 # another common subdir
]  # Add/adjust as needed. First match wins.

# Edge detector & mask settings
CANNY_LOW = 50
CANNY_HIGH = 150
GAUSS_BLUR = 1.0                 # sigma for pre-blur (0/None to skip)
MORPH_KERNEL = 3                 # opening/closing kernel (odd int)
CLAHE_ON = False                 # LSI-Lite default posture: OFF
LONG_SIDE_RESIZE = None          # None = use native size; or set e.g. 1024
# -----------------------------------------------------------------------------

def resolve_path(name: str) -> Path:
    p = Path(name)
    if p.exists():
        return p
    for root in IMAGE_ROOTS:
        cand = Path(root) / name
        if cand.exists():
            return cand
    return p  # will fail later; we let the caller handle

def load_gray(path: Path) -> np.ndarray:
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    if LONG_SIDE_RESIZE:
        h, w = img.shape[:2]
        scale = LONG_SIDE_RESIZE / max(h, w)
        if scale < 1.0:
            img = cv2.resize(img, (int(round(w*scale)), int(round(h*scale))), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if CLAHE_ON:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)
    return gray

def build_mask(gray: np.ndarray) -> np.ndarray:
    # Otsu on gray, expecting dark-on-light; small open/close
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_KERNEL, MORPH_KERNEL))
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=1)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k, iterations=1)
    return th  # 0/255

def edge_map(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    g = gray.copy()
    if GAUSS_BLUR and GAUSS_BLUR > 0:
        ksize = max(3, int(2*round(3*GAUSS_BLUR)+1))
        g = cv2.GaussianBlur(g, (ksize, ksize), GAUSS_BLUR)
    edges = cv2.Canny(g, CANNY_LOW, CANNY_HIGH)  # 0/255
    edges = cv2.bitwise_and(edges, edges, mask=mask)  # keep structural zones
    return (edges > 0).astype(np.uint8)

def largest_cc_fraction(binary_edges: np.ndarray) -> float:
    if binary_edges.sum() == 0:
        return 0.0
    num, labels = cv2.connectedComponents(binary_edges, connectivity=8)
    counts = np.bincount(labels.ravel())
    lcc = counts[1:].max() if len(counts) > 1 else 0
    return float(lcc) / float(binary_edges.sum())

def skeleton_and_graph(binary_edges: np.ndarray):
    # Skeletonize the *edges* to 1px width
    sk = skeletonize(binary_edges.astype(bool)).astype(np.uint8)
    return sk

def degree_map(skel: np.ndarray) -> np.ndarray:
    # Count 8-neighborhood degree for each skeleton pixel
    if skel.sum() == 0:
        return np.zeros_like(skel, dtype=np.uint8)
    kernel = np.ones((3,3), np.uint8)
    neighbor_count = cv2.filter2D(skel, -1, kernel, borderType=cv2.BORDER_CONSTANT)
    # neighbor_count includes self; degree = neighbors - 1
    deg = (neighbor_count - skel).astype(np.uint8)
    return deg

def path_stats_from_skeleton(skel: np.ndarray):
    """Return mean_path (avg geodesic length of segments between junctions)
       and break_per_1k (endpoints per 1000 edge px as a proxy for breaks).
    """
    total_edge = int(skel.sum())
    if total_edge == 0:
        return 0.0, 0.0

    deg = degree_map(skel)
    endpoints = ((skel == 1) & (deg == 1)).astype(np.uint8)
    junctions = ((skel == 1) & (deg >= 3)).astype(np.uint8)
    # Label segments by removing junctions, then CCs ~ segments/runs
    sk_wo_junc = skel.copy()
    sk_wo_junc[junctions.astype(bool)] = 0
    num, labels = cv2.connectedComponents(sk_wo_junc, connectivity=8)
    lengths = []
    for lab in range(1, num):
        lengths.append(int((labels == lab).sum()))
    mean_path = float(np.mean(lengths)) if lengths else float(total_edge)

    # Break proxy: many endpoints per length = fragmented structure
    n_end = int(endpoints.sum())
    break_per_1k = (n_end / max(1, total_edge)) * 1000.0
    return mean_path, break_per_1k

def orientation_coherence(gray: np.ndarray, mask: np.ndarray) -> float:
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx*gx + gy*gy)
    ang = np.arctan2(gy, gx)  # [-pi, pi]
    # Only within mask
    m = (mask > 0)
    mag = mag[m]
    ang = ang[m]
    if mag.size == 0:
        return 0.0
    # Normalize magnitudes and compute resultant vector length
    w = mag / (mag.sum() + 1e-8)
    vx = np.sum(w * np.cos(2*ang))  # 2*angle for orientation (pi-periodic)
    vy = np.sum(w * np.sin(2*ang))
    R = float(np.sqrt(vx*vx + vy*vy))  # [0,1]
    return R

def compute_metrics_for_path(p: Path):
    gray = load_gray(p)
    mask = build_mask(gray)  # 0/255
    edges = edge_map(gray, mask)
    lcc = largest_cc_fraction(edges)
    cov = float(edges.sum()) / float(gray.size)  # total pixels
    skel = skeleton_and_graph(edges)
    mean_path, break_per_1k = path_stats_from_skeleton(skel)
    coh = orientation_coherence(gray, mask)
    return {
        "lcc_pct": lcc,
        "mean_path": mean_path,
        "break_per_1k": break_per_1k,
        "edge_coverage": cov,
        "orient_coh": coh,
    }

# --- choose source dataframe for EC ---
import pandas as pd

if "base" in globals() and isinstance(base, pd.DataFrame) and not base.empty:
    df = base.copy()
elif "df_out" in globals() and isinstance(df_out, pd.DataFrame) and not df_out.empty:
    df = df_out.copy()
elif "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    # fall back to existing df if nothing else
    df = df.copy()
else:
    raise RuntimeError("Edge continuity: no suitable dataframe (base / df_out / df). Run scoring first.")

# ensure we have a frame identifier to resolve paths from
if "frame" not in df.columns:
    # try some common alternates; otherwise use index as a label
    for alt in ["frame_path", "image", "img_path"]:
        if alt in df.columns:
            df["frame"] = df[alt].astype(str)
            break
    else:
        df["frame"] = df.index.astype(str)

# ----------------------------- RUN -------------------------------------------
rows = []
missing = []
for i, row in df.reset_index(drop=True).iterrows():
    name = str(row["frame"])
    p = resolve_path(name)
    try:
        stats = compute_metrics_for_path(p)
    except Exception as e:
        stats = {"lcc_pct": np.nan, "mean_path": np.nan, "break_per_1k": np.nan,
                 "edge_coverage": np.nan, "orient_coh": np.nan}
        missing.append((name, str(e)))
    rows.append(stats)

edge_df = pd.DataFrame(rows)
df = pd.concat([df.reset_index(drop=True), edge_df], axis=1)

# Composite EC score (0–1) using cohort z-scores; clamp for readability
def z(x):
    if np.all(np.isnan(x)): return x
    mu, sd = np.nanmean(x), np.nanstd(x) + 1e-8
    return (x - mu) / sd

z_lcc   = z(df["lcc_pct"].values.astype(float))
z_path  = z(df["mean_path"].values.astype(float))
z_break = z(df["break_per_1k"].values.astype(float))
z_coh   = z(df["orient_coh"].values.astype(float))

EC = (0.4*z_lcc + 0.3*z_path - 0.2*z_break + 0.1*z_coh)
EC = (EC - np.nanmin(EC)) / (np.nanmax(EC) - np.nanmin(EC) + 1e-8)
df["EC"] = EC

# Save & show a compact summary
out_path = Path("edge_continuity_stats.csv")
df.to_csv(out_path, index=False)
print(f"✓ Edge continuity fields added to df and saved to {out_path.resolve()}")
print("\nPer-basin medians (EC & parts):")
if "basin" in df.columns:
    summary = df.groupby("basin")[["lcc_pct","mean_path","break_per_1k","edge_coverage","orient_coh","EC"]].median().round(4)
    print(summary)
else:
    print(df[["lcc_pct","mean_path","break_per_1k","edge_coverage","orient_coh","EC"]].median().round(4))

if missing:
    print("\nWarnings (unreadable files):")
    for n, e in missing[:10]:
        print(f"  - {n}: {e}")
    if len(missing) > 10:
        print(f"  (+{len(missing)-10} more)")
# -----------------------------------------------------------------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Edge Continuity — charts & exemplars (inline only) ===
# Requirements: matplotlib, numpy, pandas, opencv-python, pillow (optional)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

# If your df['frame'] entries are just filenames, these roots are searched:
IMAGE_ROOTS = ["./", "./images", "./frames"]
INCLUDE_EXEMPLARS = True   # set False to skip highest/lowest EC example images

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def resolve_path(name: str) -> Path:
    """Resolve frame name to an on-disk path, searching IMAGE_ROOTS."""
    p = Path(name)
    if p.is_file():
        return p

    for root in IMAGE_ROOTS:
        cand = Path(root) / name
        if cand.is_file():
            return cand

    # If nothing matched, just return the raw name; caller will handle errors
    return p

def iqr_series(s: pd.Series):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan, np.nan, np.nan
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75])
    return q2, q1, q3

# ------------------------------------------------------------
# 0) basic sanity
# ------------------------------------------------------------

if "df" not in globals() or df.empty:
    raise RuntimeError("EC charts: df is missing or empty. Run the EC metrics cell first.")

required_cols = {"EC", "lcc_pct", "mean_path", "break_per_1k",
                 "edge_coverage", "orient_coh", "basin", "frame"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise RuntimeError(f"EC charts: df is missing required columns: {sorted(missing_cols)}")

# ------------------------------------------------------------
# 1) EC by basin (median + IQR whiskers)
# ------------------------------------------------------------

grouped = df.groupby("basin")["EC"]
meds = grouped.median()
q1_q3 = grouped.quantile([0.25, 0.75]).unstack()

iqr_low_raw = q1_q3[0.25] - meds
iqr_high_raw = q1_q3[0.75] - meds

# Clamp to zero so Matplotlib never sees negative error bars
iqr_low = np.clip(iqr_low_raw.values, 0.0, None)
iqr_high = np.clip(iqr_high_raw.values, 0.0, None)

plt.figure(figsize=(6, 4))
plt.title("Edge Continuity by Basin (median with IQR whiskers)")
plt.errorbar(
    x=meds.index.astype(str),
    y=meds.values,
    yerr=[iqr_low, iqr_high],
    fmt="o",
    capsize=6,
)
plt.xlabel("Basin")
plt.ylabel("EC (0–1)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 2) Scatter: EC vs rupture_rho (packing)
# ------------------------------------------------------------

plt.figure(figsize=(6, 4))
plt.scatter(df["rupture_rho"], df["EC"])
plt.xlabel("ρ_r (packing)")
plt.ylabel("EC")
plt.title("EC vs Packing (ρ_r)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3) Scatter: EC vs void_ratio (r_v)
# ------------------------------------------------------------

plt.figure(figsize=(6, 4))
plt.scatter(df["void_ratio"], df["EC"])
plt.xlabel("r_v (void)")
plt.ylabel("EC")
plt.title("EC vs Void (r_v)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4) Summary table (medians per basin)
# ------------------------------------------------------------

summary = (
    df.groupby("basin")[["lcc_pct", "mean_path", "break_per_1k",
                         "edge_coverage", "orient_coh", "EC"]]
      .median()
      .round(4)
)
print("\n=== Per-basin medians ===")
print(summary)

# ------------------------------------------------------------
# 5) Optional exemplars: highest & lowest EC (single-plot each)
# ------------------------------------------------------------

if INCLUDE_EXEMPLARS and not df.empty:
    hi = df.sort_values("EC", ascending=False).iloc[0]
    lo = df.sort_values("EC", ascending=True).iloc[0]

    for row, label in [(hi, "Highest EC (most connected structure)"),
                       (lo, "Lowest EC (most fragmented structure)")]:
        path = resolve_path(str(row["frame"]))
        title = (
            f"{label}\n"
            f"EC={row['EC']:.3f} | LCC={row['lcc_pct']:.3f} | "
            f"path={row['mean_path']:.1f} | breaks/1k={row['break_per_1k']:.1f}"
        )

        plt.figure(figsize=(5, 5))
        if path.is_file():
            img = Image.open(path).convert("RGB")
            plt.imshow(img)
            plt.axis("off")
            plt.title(title, fontsize=9)
        else:
            plt.axis("off")
            plt.text(
                0.5,
                0.5,
                f"Image not found:\n{path}",
                ha="center",
                va="center",
                fontsize=10,
            )
        plt.tight_layout()
        plt.show()

# ------------------------------------------------------------
# 6) Correlations (quick sanity)
# ------------------------------------------------------------

def corr_x_y(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

c_rho = corr_x_y(df["rupture_rho"], df["EC"])
c_rv = corr_x_y(df["void_ratio"], df["EC"])

print("\n=== Correlations (sanity check) ===")
print(f"Pearson corr: EC vs ρ_r  = {c_rho: .3f}")
print(f"Pearson corr: EC vs r_v  = {c_rv: .3f}")

print("\nEC charts & exemplars: inline display complete.")

**Technical Description (For Researchers)**

The System

LSI-lite evaluates compositional structure through three primitives:

- Δx (off-center gravity) — measures horizontal displacement from geometric center
- rᵥ (void ratio) — quantifies negative space distribution
- ρᵣ (rupture/mark energy) — captures edge density in subject halo

**The Intelligence**

**Adaptive mask selection:** System compares grayscale Otsu and LAB k-means segmentation quality. When color provides superior subject isolation (area fraction 0.03–0.65), color-derived Δx and rᵥ replace grayscale measurements in scoring. This demonstrates intelligent routing to the best measurement source.

**Fallback robustness:** If color segmentation fails or produces poor masks, system automatically defaults to grayscale processing.

**Quality audits:** Additional color/tonal metrics (delta measurements, tonal span, separability) serve as non-gating diagnostic badges that flag potential issues without affecting pass/fail.

**Consequence**
- Detects systematic biases current metrics miss (radial collapse, compression patterns)
- Operates at compositional level, not pixel similarity
- Three profiles (Figure, Mark-Making, Landscape) with tuned band guards
- Can identify "pre-failure" states where images are structurally unstable


**Key Differentiators**
What LSI-lite IS:

- Compositional structure measurement (not aesthetic judgment)
- Detects algorithmic defaults vs intentional composition
- Complementary to existing metrics (CLIP, FID, etc.)
- Built from 25+ years of figure drawing/visual practice

What LSI-lite IS NOT:

- Style police or aesthetic arbiter
- Pixel-level quality assessment
- Subjective preference scoring
- Replacement for human evaluation

# Questions and Answers

Q: How is this different from CLIP or aesthetic scoring?

A: CLIP measures semantic similarity and aesthetic scorers predict human preferences. LSI-lite measures structural primitives — whether an image exhibits compositional coherence independent of content or style. You can have low CLIP score but high LSI (intentional avant-garde), or high CLIP but low LSI (pretty but structurally collapsed).

Q: What's a "pre-failure state"?

A: When images pass pixel-level quality checks but show structural instability — measurements clustering near band edges, high void with low detail, extreme displacement with minimal mark energy. These correlate with prompting that forces the model away from its training distribution.

Q: Why three profiles?

A: Different compositional genres have different stability regions. Figure compositions naturally center around 0.15–0.25 displacement; landscapes tolerate 0.05–0.95; mark-making can go to extremes if rupture energy supports it. Single-threshold systems miss this genre-specific behavior.

Q: How does color help?

A: LAB k-means often gets better figure/ground separation than grayscale Otsu, especially on subtle subjects or complex backgrounds. When color segmentation quality exceeds threshold, system uses color-derived measurements. If color fails, automatic fallback to gray. Demonstrates practical robustness.

Systematic bias detection:

GPT shows reproducible right-displacement bias (median Δx ≈ 0.18)
Midjourney shows left-compression patterns (corridor_90 < 0.45)
Both measurable across 100+ image batches


#Pre-failure detection:

- Images with LSI 45–55 show likely structural instability
- Band edge clustering predicts prompt refinement needs
- Void/rupture correlation reveals composition collapse


#Complementarity to existing metrics:

- Cases where CLIP=high but LSI=low (aesthetic but structurally centered)
- Cases where FID=low but LSI=low (pixel-similar but compositionally collapsed)
- LSI adds orthogonal dimension to evaluation space

 Diagnostics & Debugging ← Optional, for power users

In [ ]:
# Diagnostic Cell - Run this BEFORE scoring to see what's happening with masks

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import glob

# Pick first 3-5 images to diagnose
test_paths = sorted(glob.glob("/content/images/*"))[:5]

print("=== MASK SELECTION DIAGNOSTICS ===\n")

for path in test_paths:
    print(f"\n{'='*60}")
    print(f"Image: {path.split('/')[-1]}")
    print('='*60)

    # Load
    gray, rgb = load_image_robust(path)

    # Gray mask
    gray_mask = foreground_mask(gray, CONFIG)
    f_gray = (gray_mask > 0).sum() / gray_mask.size
    gray_ok = (0.03 <= f_gray <= 0.65)

    # Color mask
    try:
        color_mask = color_mask_via_lab_kmeans(rgb, k=3, morph_kernel=CONFIG["preprocessing"]["morph_kernel"])
        f_color = (color_mask > 0).sum() / color_mask.size
        color_ok = (0.03 <= f_color <= 0.65)
        color_status = "ok"
    except Exception as e:
        color_mask = None
        f_color = 0.0
        color_ok = False
        color_status = f"error: {str(e)[:50]}"

    # Report
    print(f"\nGRAY MASK:")
    print(f"  Fill: {f_gray:.3f} ({f_gray*100:.1f}%)")
    print(f"  Valid: {gray_ok} (range: 0.03-0.65)")

    print(f"\nCOLOR MASK:")
    print(f"  Status: {color_status}")
    print(f"  Fill: {f_color:.3f} ({f_color*100:.1f}%)")
    print(f"  Valid: {color_ok} (range: 0.03-0.65)")

    # Decision
    if color_ok and not gray_ok:
        winner = "COLOR (gray out of range)"
    elif gray_ok and not color_ok:
        winner = "GRAY (color out of range)"
    elif gray_ok and color_ok:
        winner = "GRAY (both valid, gray default)"  # ← This is the problem
    else:
        winner = "GRAY (both invalid, fallback)"

    print(f"\nDECISION: {winner}")

    # Visualize
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(gray, cmap="gray")
    axes[1].set_title("Grayscale")
    axes[1].axis("off")

    axes[2].imshow(gray_mask, cmap="gray")
    axes[2].set_title(f"Gray Mask\nfill={f_gray:.2f}, valid={gray_ok}")
    axes[2].axis("off")

    if color_mask is not None:
        axes[3].imshow(color_mask, cmap="gray")
        axes[3].set_title(f"Color Mask\nfill={f_color:.2f}, valid={color_ok}")
    else:
        axes[3].text(0.5, 0.5, "Color Failed", ha="center", va="center", fontsize=14)
        axes[3].set_title("Color Mask")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print("\nIf you see 'GRAY (both valid, gray default)' repeatedly,")
print("that's why color never wins. Both masks work, but gray")
print("is hardcoded as the default.")
print("\nFIX: Use the improved mask selection function that")
print("compares quality or prefers color when both are valid.")

In [ ]:
print(df['mask_source'].value_counts())
print(f"Color usage: {(df['mask_source'] == 'color').sum() / len(df) * 100:.1f}%")